In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

# -------------------------------
# Step 1: 讀取 NBA 2009 選秀前 30
# -------------------------------
draft_url = "https://www.basketball-reference.com/draft/NBA_2003.html"
dfs = pd.read_html(draft_url, header=[0, 1])
df = dfs[0]

# 合併多層欄名
df.columns = ['_'.join(col).strip() for col in df.columns.values]

# 取前 30 名
df_r1 = df.head(30)

# 只保留順位、球員、大學
df_r1 = df_r1[['Unnamed: 1_level_0_Pk', 'Round 1_Player', 'Round 1_College']]

# 去掉大學是空白的
df_r1 = df_r1.dropna(subset=['Round 1_College'])

# 過濾掉國際球員
exclude_keywords = r'Spain|France|Turkey|China|Serbia|Croatia|Lithuania|Brazil|Slovenia|Argentina|Germany|Italy|Greece|Nigeria|Senegal'
df_r1_us = df_r1[~df_r1['Round 1_College'].str.contains(exclude_keywords, case=False, na=False)]

# 最後只要前 30 的美國大學球員
players = df_r1_us.reset_index(drop=True)
print("✅ 前 30 美國大學球員：")
print(players)

# -------------------------------
# Step 2: 定義函式抓 Sports-Reference 數據
# -------------------------------
def get_player_stats(name):
    """
    嘗試用不同尾碼搜尋球員 NCAA 數據頁
    """
    clean_name = name.lower().replace('.', '').replace("'", '').replace(' ', '-')
    for i in range(1, 9):  # 嘗試尾碼 1~8
        url = f"https://www.sports-reference.com/cbb/players/{clean_name}-{i}.html"
        try:
            tables = pd.read_html(url)
            df_stats = tables[0]
            if "Season" in df_stats.columns:
                # 只抓 2008-09 賽季（球員最後一年 NCAA）
                row = df_stats[df_stats["Season"] == "2002-03"]
                if not row.empty:
                    print(f"✅ {name}-{i}: 成功擷取")
                    return row[['Pos','FG','FGA','ORB','DRB','TRB','AST','STL','BLK','TOV','PF','PTS','Season']]
        except Exception:
            pass
        time.sleep(1.5)  # 避免請求過快
    print(f"❌ {name}: 找不到資料")
    return None

# -------------------------------
# Step 3: 逐一抓取並存 Excel
# -------------------------------
results = {}
for name in players['Round 1_Player']:
    print(f"處理：{name}")
    stats = get_player_stats(name)
    if stats is not None:
        results[name] = stats
    else:
        print(f"⚠ 找不到資料：{name}")
    time.sleep(1.5)

# 儲存到 Excel
with pd.ExcelWriter("newstats2003.xlsx") as writer:
    for name, stat in results.items():
        stat.to_excel(writer, sheet_name=name[:31], index=False)  # Excel 限制工作表名 <= 31 字

print("🎉 完成！結果已存成 newstats2007.xlsx")


✅ 前 30 美國大學球員：
   Unnamed: 1_level_0_Pk   Round 1_Player              Round 1_College
0                      3  Carmelo Anthony                     Syracuse
1                      4       Chris Bosh                 Georgia Tech
2                      5      Dwyane Wade                    Marquette
3                      6      Chris Kaman  Central Michigan University
4                      7     Kirk Hinrich                       Kansas
5                      8        T.J. Ford                        Texas
6                      9    Mike Sweetney                   Georgetown
7                     10     Jarvis Hayes                      Georgia
8                     12    Nick Collison                       Kansas
9                     13     Marcus Banks                         UNLV
10                    14     Luke Ridnour                       Oregon
11                    15     Reece Gaines                   Louisville
12                    16        Troy Bell               Boston

In [9]:
import pandas as pd
import os

# 讀取原始 Excel 檔案（每個球員一個 sheet）
excel_path = 'newstats2003.xlsx'
xlsx = pd.ExcelFile(excel_path)

# 儲存所有合併後的資料
combined_data = []

for sheet_name in xlsx.sheet_names:
    if sheet_name == "All Players":  # 跳過總表
        continue
    df = pd.read_excel(xlsx, sheet_name=sheet_name)
    df['Player'] = sheet_name  # 將 sheet 名當作 Player 名
    combined_data.append(df)

# 合併所有 sheet 的資料
final_df = pd.concat(combined_data, ignore_index=True)

# 儲存為新的 Excel 檔案，只包含一個 sheet
output_path = 'combined_college_stats2003.xlsx'
final_df.to_excel(output_path, index=False)
print(f"✅ 大學數據已合併並儲存為 {output_path}")


✅ 大學數據已合併並儲存為 combined_college_stats2003.xlsx


In [22]:
import pandas as pd

draft_url = "https://www.basketball-reference.com/draft/NBA_2003.html"
dfs = pd.read_html(draft_url, header=[0, 1])
df_ncaa = dfs[0]

# 攤平欄位
df_ncaa.columns = ['_'.join(col).strip() for col in df_ncaa.columns.values]

# 把順位轉數字
df_ncaa['Pick'] = pd.to_numeric(df_ncaa['Unnamed: 1_level_0_Pk'], errors='coerce')

# 篩選第一輪（Pick <= 30）
df_r1 = df_ncaa[df_ncaa['Pick'] <= 30]

# 去掉沒有大學的球員
df_r1 = df_r1.dropna(subset=['Round 1_College'])

# 過濾掉國際球員
exclude_keywords = r'Spain|France|Turkey|China|Serbia|Croatia|Lithuania|Brazil|Slovenia|Argentina|Germany|Italy|Greece|Nigeria|Senegal'
df_r1_us = df_r1[~df_r1['Round 1_College'].str.contains(exclude_keywords, case=False, na=False)]

# 提取需要的欄位
players_df = df_r1_us[['Round 1_Player', 'Round 1_College', 'Pick', 'Unnamed: 2_level_0_Tm']].reset_index(drop=True)
players_df.columns = ['Player', 'College', 'Pick', 'Team']

print(players_df)




             Player                      College  Pick Team
0   Carmelo Anthony                     Syracuse   3.0  DEN
1        Chris Bosh                 Georgia Tech   4.0  TOR
2       Dwyane Wade                    Marquette   5.0  MIA
3       Chris Kaman  Central Michigan University   6.0  LAC
4      Kirk Hinrich                       Kansas   7.0  CHI
5         T.J. Ford                        Texas   8.0  MIL
6     Mike Sweetney                   Georgetown   9.0  NYK
7      Jarvis Hayes                      Georgia  10.0  WAS
8     Nick Collison                       Kansas  12.0  SEA
9      Marcus Banks                         UNLV  13.0  MEM
10     Luke Ridnour                       Oregon  14.0  SEA
11     Reece Gaines                   Louisville  15.0  ORL
12        Troy Bell               Boston College  16.0  BOS
13       David West                       Xavier  18.0  NOH
14    Dahntay Jones                         Duke  20.0  BOS
15       Brian Cook                     

In [23]:
from nba_api.stats.static import players
from nba_api.stats.endpoints import commonplayerinfo
import pandas as pd

# 查詢 NBA ID
def find_nba_id(name):
    match = players.find_players_by_full_name(name)  # 這裡用 players
    if match:
        return match[0]['id']
    return None

# 假設你已經有 players_df 這個 DataFrame
players_df['Player_ID'] = players_df['Player'].apply(find_nba_id)
players_df = players_df.dropna(subset=['Player_ID'])  # 移除找不到 ID 的球員
players_df['Player_ID'] = players_df['Player_ID'].astype(int)


In [24]:
all_stats = []

for _, row in tqdm(players_df.iterrows(), total=len(players_df)):
    try:
        gamelog = playergamelog.PlayerGameLog(
            player_id=row['Player_ID'],
            season='2003-04',
            season_type_all_star='Regular Season'
        ).get_data_frames()[0]

        if not gamelog.empty:
            season_totals = gamelog.describe().loc[['mean']]  # 平均值
            season_totals['Player'] = row['Player']
            season_totals['Player_ID'] = row['Player_ID']
            season_totals['College'] = row['College']
            season_totals['Pick'] = row['Pick']
            season_totals['Team'] = row['Team']
            all_stats.append(season_totals)

        time.sleep(0.8)  # 避免 API 過載
    except Exception as e:
        print(f"Failed to fetch {row['Player']}: {e}")
        continue

# 合併所有球員數據
nba_rookie_stats = pd.concat(all_stats, ignore_index=True)


100%|██████████| 16/16 [00:00<00:00, 3441.83it/s]

Failed to fetch Carmelo Anthony: name 'playergamelog' is not defined
Failed to fetch Chris Bosh: name 'playergamelog' is not defined
Failed to fetch Dwyane Wade: name 'playergamelog' is not defined
Failed to fetch Chris Kaman: name 'playergamelog' is not defined
Failed to fetch Kirk Hinrich: name 'playergamelog' is not defined
Failed to fetch T.J. Ford: name 'playergamelog' is not defined
Failed to fetch Jarvis Hayes: name 'playergamelog' is not defined
Failed to fetch Nick Collison: name 'playergamelog' is not defined
Failed to fetch Marcus Banks: name 'playergamelog' is not defined
Failed to fetch Luke Ridnour: name 'playergamelog' is not defined
Failed to fetch Reece Gaines: name 'playergamelog' is not defined
Failed to fetch Troy Bell: name 'playergamelog' is not defined
Failed to fetch David West: name 'playergamelog' is not defined
Failed to fetch Dahntay Jones: name 'playergamelog' is not defined
Failed to fetch Brian Cook: name 'playergamelog' is not defined
Failed to fetch Jos

ValueError: No objects to concatenate

In [19]:
nba_rookie_stats.to_csv('nba_rookie_stats_2003_04.csv', index=False)


NameError: name 'nba_rookie_stats' is not defined

In [171]:
import pandas as pd

# 讀取職業 NBA 數據
nba_df = pd.read_csv("nba_rookie_stats_2009_10.csv")

# 讀取大學數據
college_df = pd.read_excel("combined_college_stats2009.xlsx")

# 確保名稱欄位正確
if 'Player' not in nba_df.columns or 'Player' not in college_df.columns:
    raise ValueError("兩個檔案都需要有 'Player' 欄位才能合併")

# 合併
merged_df = pd.merge(nba_df, college_df, on='Player', how='left')

# 讀取 ALL_STATS.xlsx
try:
    all_stats = pd.read_excel("ALL_STATS.xlsx")
    # 把新資料往下接
    final_df = pd.concat([all_stats, merged_df], ignore_index=True)
except FileNotFoundError:
    # 如果沒有 ALL_STATS.xlsx，就直接用 merged_df
    final_df = merged_df

# 覆蓋儲存 ALL_STATS.xlsx
final_df.to_excel("ALL_STATS.xlsx", index=False)

print("✅ 整合完成，已更新 ALL_STATS.xlsx")


✅ 整合完成，已更新 ALL_STATS.xlsx


In [207]:
import pandas as pd
from nba_api.stats.static import players as nba_players
from nba_api.stats.endpoints import playergamelog
import time

# 讀取資料
df = pd.read_excel("ALL_STATS_with_WS.xlsx")

# 取得所有 NBA 官方球員清單
player_dict = nba_players.get_players()

# 找 Player_ID
def get_player_id(name):
    for p in player_dict:
        if p["full_name"].lower() == str(name).lower():
            return p["id"]
    return None

# 抓新秀前三年的平均得分
def get_avg_pts(player_id):
    try:
        gamelog = playergamelog.PlayerGameLog(
            player_id=player_id,
            season="ALL",
            season_type_all_star="Regular Season"
        ).get_data_frames()[0]

        # 拿到所有出現過的賽季
        seasons = sorted(gamelog["SEASON_ID"].unique())

        if len(seasons) == 0:
            print(f"⚠️ {player_id}: 沒有比賽紀錄")
            return None

        # 取第一個賽季當作新秀年
        rookie_seasons = seasons[:3]  # 前三年
        subset = gamelog[gamelog["SEASON_ID"].isin(rookie_seasons)]

        if not subset.empty:
            avg_pts = subset["PTS"].mean()
            games = len(subset)
            print(f"✅ {player_id}: {rookie_seasons} 共 {games} 場, 平均 {avg_pts:.2f} 分")
            return avg_pts
        else:
            print(f"⚠️ {player_id}: 找不到前三年比賽紀錄")
            return None

    except Exception as e:
        print(f"❌ 抓取失敗 (ID={player_id}): {e}")
        return None




# 建立 Player_ID 欄位（只對 DraftYear ≤ 2022 的球員）
df["Player_ID"] = df["Player"].apply(get_player_id)

# 新增欄位存平均得分
df["AvgPTS_3yrs"] = None

# 篩選 DraftYear ≤ 2022
df_filtered = df[df["DraftYear"] <= 2022]

# 逐一處理
for idx, row in df_filtered.iterrows():
    if pd.notna(row["Player_ID"]) and pd.notna(row["DraftYear"]):
        avg_pts = get_avg_pts(int(row["Player_ID"]))
        df.at[idx, "AvgPTS_3yrs"] = avg_pts
    else:
        print(f"⚠️ 跳過 {row['Player']} (缺少 Player_ID 或 DraftYear)")

    time.sleep(0.6)  # 避免 API 過載

# 存回 Excel
df.to_excel("ALL_STATS_with_WS.xlsx", index=False)
print("🏁 已完成，結果已寫回 ALL_STATS_with_WS.xlsx")


✅ 1631094: ['22022', '22023', '22024'] 共 198 場, 平均 22.38 分
✅ 1631095: ['22022', '22023', '22024'] 共 212 場, 平均 12.98 分
✅ 1631099: ['22022', '22023', '22024'] 共 233 場, 平均 13.26 分
✅ 1631093: ['22022', '22023', '22024'] 共 181 場, 平均 16.12 分
✅ 1631097: ['22022', '22023', '22024'] 共 209 場, 平均 15.86 分
✅ 1631110: ['22022', '22023', '22024'] 共 184 場, 平均 11.37 分
✅ 1631114: ['22022', '22023', '22024'] 共 215 場, 平均 18.14 分
✅ 1631105: ['22022', '22023', '22024'] 共 206 場, 平均 11.51 分
✅ 1630534: ['22022', '22023', '22024'] 共 201 場, 平均 7.91 分
✅ 1631109: ['22022', '22023', '22024'] 共 106 場, 平均 12.26 分
✅ 1631100: ['22022', '22023'] 共 92 場, 平均 7.47 分
✅ 1631106: ['22022', '22023', '22024'] 共 161 場, 平均 10.32 分
✅ 1631207: ['22022', '22023', '22024'] 共 170 場, 平均 3.48 分
✅ 1631222: ['22022', '22023', '22024'] 共 136 場, 平均 6.93 分
✅ 1631222: ['22022', '22023', '22024'] 共 136 場, 平均 6.93 分
✅ 1631222: ['22022', '22023', '22024'] 共 136 場, 平均 6.93 分
✅ 1631103: ['22022', '22023', '22024'] 共 188 場, 平均 8.50 分
✅ 1631128: ['2

In [208]:
pip install fuzzywuzzy[speedup]

In [212]:
import pandas as pd
from fuzzywuzzy import process

# 讀取 Excel
df = pd.read_excel(file_path)

# 確保 DraftYear 是數字
df["DraftYear"] = pd.to_numeric(df["DraftYear"], errors="coerce")

# 篩選 DraftYear <= 2022
df_filtered = df[df["DraftYear"] >= 2023].copy()

# 建立暫存 dict
ws_dict = {}

for year in sorted(df_filtered["DraftYear"].unique()):
    print(f"📥 抓取 {year} 年的新秀球員 WS / WS48...")

    url = f"https://www.basketball-reference.com/leagues/NBA_{year+1}_advanced.html"
    try:
        bbr_df = pd.read_html(url, header=0)[0]

        # 清理資料
        bbr_df = bbr_df[bbr_df["Rk"] != "Rk"]
        bbr_df["Player"] = (
            bbr_df["Player"].str.replace(r"[*]", "", regex=True).str.strip()
        )
        bbr_df.columns = bbr_df.columns.astype(str).str.strip()

        # 只留需要欄位，並改名避免衝突
        bbr_df = bbr_df[["Player", "WS", "WS/48"]].copy()
        bbr_df = bbr_df.rename(columns={"WS": "WS_bbr", "WS/48": "WS48_bbr"})

        ws_dict[year] = bbr_df
        print(f"✅ {year} 完成，共 {len(bbr_df)} 筆球員")

    except Exception as e:
        print(f"❌ {year} 抓取失敗: {e}")

# 合併資料
df_result = df.copy()

for year, bbr_df in ws_dict.items():
    mask = df_result["DraftYear"] == year
    df_year = df_result[mask].copy()

    # 嘗試直接 merge
    merged = df_year.merge(bbr_df, on="Player", how="left")

    # 確保欄位存在
    if "WS_bbr" not in merged.columns:
        merged["WS_bbr"] = None
    if "WS48_bbr" not in merged.columns:
        merged["WS48_bbr"] = None

    # 對於 NaN 的部分，使用 fuzzy matching
    for idx, row in merged[merged["WS_bbr"].isna()].iterrows():
        player_name = row["Player"]
        result = process.extractOne(player_name, bbr_df["Player"])

        if result:
            if len(result) == 2:
                match, score = result
            elif len(result) == 3:
                match, score, _ = result  # 有些版本會多一個 index
            else:
                continue

            if score >= 90:  # 相似度閾值
                stats_row = bbr_df[bbr_df["Player"] == match].iloc[0]
                merged.at[idx, "WS_bbr"] = stats_row["WS_bbr"]
                merged.at[idx, "WS48_bbr"] = stats_row["WS48_bbr"]
                print(f"🔄 模糊匹配：{player_name} → {match} (score={score})")

    # 更新回 df_result
        # 更新回 df_result
    ws_map = dict(zip(merged["Player"], merged["WS_bbr"]))
    ws48_map = dict(zimport pandas as pd


# 存回 Excel（覆蓋原檔案）
df_result.to_excel(file_path, index=False)
print(f"💾 已更新並存回 {file_path}")


SyntaxError: invalid syntax. Perhaps you forgot a comma? (2116548778.py, line 78)

In [230]:
import pandas as pd
import requests, time, random
from bs4 import BeautifulSoup, Comment

# 載入 Excel
file_path = "ALL_STATS_with_WS.xlsx"
df = pd.read_excel(file_path)

# 建立要新增的欄位（避免 KeyError）
advanced_cols = ["WS", "WS/40", "3PAr", "FTr", "PProd"]

for col in advanced_cols:
    if col not in df.columns:
        df[col] = None

def get_college_advanced(player_name):
    """
    嘗試抓取某球員最後一年的 Advanced 數據
    """
    slug = player_name.lower().replace(".", "").replace("'", "").replace(" ", "-")
    headers = {"User-Agent": "Mozilla/5.0"}
    
    for i in range(1, 6):  # 嘗試 -1 到 -5
        url = f"https://www.sports-reference.com/cbb/players/{slug}-{i}.html"
        try:
            r = requests.get(url, headers=headers, timeout=15)
            if r.status_code != 200:
                continue

            soup = BeautifulSoup(r.text, "html.parser")

            # advanced table 在 HTML 註解裡
            comments = soup.find_all(string=lambda text: isinstance(text, Comment))
            adv_table = None
            for c in comments:
                if "players_advanced" in c:
                    adv_table = BeautifulSoup(c, "html.parser").find("table", {"id": "players_advanced"})
                    break

            if adv_table is None:
                continue

            df_adv = pd.read_html(str(adv_table))[0]

            # 過濾最後一季（可能是轉學，要取最後一行非合計）
            last_season = df_adv[df_adv["Season"].str.contains(r"\d{4}-\d{2}", na=False)].iloc[-1]
            
            return {col: last_season.get(col, None) for col in advanced_cols}

        except Exception as e:
            print(f"⚠️ {player_name}-{i} 抓取失敗: {e}")
            continue

    return None

# 逐一抓取
not_found = []
for idx, row in df.iterrows():
    player = row["Player"]
    adv_stats = get_college_advanced(player)

    if adv_stats:
        for col, val in adv_stats.items():
            df.at[idx, col] = val
        print(f"✅ {player}: 已抓到 Advanced 數據")
    else:
        not_found.append(player)
        print(f"❌ {player}: 找不到 Advanced 表")

    # 隨機延遲，避免被封
    time.sleep(random.uniform(6, 10))

    # 每 10 人暫停 & 存檔一次
    if (idx + 1) % 10 == 0:
        df.to_excel(file_path, index=False)
        print(f"💾 已存前 {idx+1} 位，休息 60 秒...")
        time.sleep(60)

# 最後存檔
df.to_excel(file_path, index=False)

# 存下找不到的球員
with open("not_found_players.txt", "w", encoding="utf-8") as f:
    for p in not_found:
        f.write(p + "\n")

print("🎉 全部完成")


C:\Users\sara9\AppData\Local\Temp\ipykernel_17080\3064292356.py:43: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_adv = pd.read_html(str(adv_table))[0]


✅ Reed Sheppard: 已抓到 Advanced 數據


C:\Users\sara9\AppData\Local\Temp\ipykernel_17080\3064292356.py:43: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_adv = pd.read_html(str(adv_table))[0]


✅ Stephon Castle: 已抓到 Advanced 數據


C:\Users\sara9\AppData\Local\Temp\ipykernel_17080\3064292356.py:43: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_adv = pd.read_html(str(adv_table))[0]


...

✅ Tony Allen: 已抓到 Advanced 數據


C:\Users\sara9\AppData\Local\Temp\ipykernel_17080\3064292356.py:43: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_adv = pd.read_html(str(adv_table))[0]


✅ Kevin Martin: 已抓到 Advanced 數據


C:\Users\sara9\AppData\Local\Temp\ipykernel_17080\3064292356.py:43: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_adv = pd.read_html(str(adv_table))[0]


✅ David Harrison: 已抓到 Advanced 數據
🎉 全部完成


In [ ]:
import pandas as pd

# 讀取 Kaggle 資料
adv = pd.read_parquet("advanced.parq")
tot = pd.read_parquet("total.parq")

# season 轉成年份
adv["Year"] = adv["season"].str[:4].astype(int)
tot["Year"] = tot["season"].str[:4].astype(int)

# -----------------------------
# 篩選 2000–2015 年進入 NBA 的球員
# 方法：找出球員第一次出現的年份 (Year)，視為 DraftYear
first_year = adv.groupby("Player")["Year"].min().reset_index()
first_year.rename(columns={"Year": "DraftYear"}, inplace=True)

players_2000_2015 = first_year[(first_year["DraftYear"] >= 2000) & (first_year["DraftYear"] <= 2015)]

# -----------------------------
# 生涯 WS / PER / BPM
career_stats = adv.groupby("Player").agg({
    "WS": "sum",
    "PER": "mean",
    "BPM": "mean"
}).reset_index()
career_stats.rename(columns={"WS": "Career_WS", "PER": "Career_PER", "BPM": "Career_BPM"}, inplace=True)

# -----------------------------
# 新秀年進階數據
rookie_adv = adv.merge(players_2000_2015, on="Player")
rookie_adv = rookie_adv[rookie_adv["Year"] == rookie_adv["DraftYear"]]
rookie_adv = rookie_adv[["Player", "PER", "WS", "BPM", "VORP"]]
rookie_adv.rename(columns={
    "PER": "Rookie_PER",
    "WS": "Rookie_WS",
    "BPM": "Rookie_BPM",
    "VORP": "Rookie_VORP"
}, inplace=True)

# -----------------------------
# 新秀年初階數據
rookie_tot = tot.merge(players_2000_2015, on="Player")
rookie_tot = rookie_tot[rookie_tot["Year"] == rookie_tot["DraftYear"]]
rookie_tot = rookie_tot[["Player", "G", "MP", "PTS", "TRB", "AST", "STL", "BLK", "FG%", "3P%", "FT%"]]
rookie_tot.rename(columns={
    "G": "Rookie_G",
    "MP": "Rookie_MP",
    "PTS": "Rookie_PTS",
    "TRB": "Rookie_TRB",
    "AST": "Rookie_AST",
    "STL": "Rookie_STL",
    "BLK": "Rookie_BLK",
    "FG%": "Rookie_FG%",
    "3P%": "Rookie_3P%",
    "FT%": "Rookie_FT%"
}, inplace=True)

# -----------------------------
# 整合
final = players_2000_2015.merge(career_stats, on="Player", how="left")
final = final.merge(rookie_adv, on="Player", how="left")
final = final.merge(rookie_tot, on="Player", how="left")

# -----------------------------
# 輸出
final.to_excel("NBA_Stats_All_2000_2015.xlsx", index=False)

print("完成 ✅ 已輸出 NBA_Stats_All_2000_2015.xlsx")


In [6]:
import pandas as pd

# 讀取生涯數據檔
df = pd.read_excel("NBA_Stats_All_2000_2015.xlsx")

# 讀取 Kaggle 的 roster 資料
roster = pd.read_parquet("roster.parq")

# roster 裡應該有 ['player', 'season', 'position'] 之類的欄位
# 我們取每個球員出現最多的 position 當作主要位置
pos_map = roster.groupby("Player")["Pos"].agg(lambda x: x.mode()[0]).reset_index()

# 合併位置到生涯數據檔
df = df.merge(pos_map, left_on="Player", right_on="Player", how="left").drop(columns=["Player"])

# 改名比較清楚
df = df.rename(columns={"Pos": "Pos"})

# 存回 Excel
df.to_excel("NBA_Stats_All_2000_2015_with_Pos.xlsx", index=False)

print("完成！已新增 Pos 欄位並輸出到 NBA_Stats_All_2000_2015_with_Pos.xlsx")


完成！已新增 Pos 欄位並輸出到 NBA_Stats_All_2000_2015_with_Pos.xlsx


In [1]:
import pandas as pd

roster = pd.read_parquet("average.parq")
print(roster.columns.tolist())
print(roster.head())


['Rk', 'Player', 'Age', 'G', 'GS', 'MP', 'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', 'ORB', 'DRB', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'PF', 'PTS', 'team', 'season', 'team_retcon']
   Rk              Player  Age   G  GS    MP   FG   FGA    FG%   3P  ...  TRB  \
0   1        Isaiah Rider   23  75  67  35.3  7.4  16.7  0.447  1.9  ...  3.3   
1   2  Christian Laettner   25  81  80  34.2  5.6  11.4  0.489  0.2  ...  7.6   
2   3           Doug West   27  71  65  32.8  4.9  10.7  0.461  0.2  ...  3.2   
3   4       Tom Gugliotta   25  31  17  32.8  5.2  11.5  0.454  0.9  ...  7.2   
4   5          Sean Rooks   25  80  70  30.1  3.6   7.7  0.470  0.0  ...  6.1   

   AST  STL  BLK  TOV   PF   PTS  team     season  team_retcon  
0  3.3  0.9  0.3  3.1  2.6  20.4   MIN  1995/1996          MIN  
1  2.9  1.2  1.1  2.8  3.7  16.3   MIN  1995/1996          MIN  
2  2.6  0.9  0.3  1.8  3.5  12.9   MIN  1995/1996          MIN  
3  4.5  2.0  0.9  2.6  2.8  1

In [ ]:
import pandas as pd

# 讀檔案
adv = pd.read_parquet("advanced.parq")
avg = pd.read_parquet("average.parq")
roster = pd.read_parquet("roster.parq")

# season 格式是 "1996/1997"，先取前4碼轉成 int
roster['season_start'] = roster['season'].str[:4].astype(int)

# 推算 DraftYear = 最早出現的賽季 - 1
roster['DraftYear'] = roster.groupby("Player")['season_start'].transform('min') - 1

# 篩選 1995–2015 DraftYear
players_sel = roster[['Player','DraftYear']].drop_duplicates()
players_sel = players_sel[(players_sel['DraftYear'] >= 1995) & (players_sel['DraftYear'] <= 2015)]

# 生涯數據
career_adv = adv.groupby("Player").agg({
    "WS": "sum",
    "PER": "mean",
    "BPM": "mean"
}).reset_index().rename(columns={"WS":"Career_WS","PER":"Career_PER","BPM":"Career_BPM"})

# 菜鳥年 (DraftYear+1)
rookie_adv = adv.merge(players_sel, on="Player", how="inner")
rookie_adv['season_start'] = rookie_adv['season'].str[:4].astype(int)
rookie_adv = rookie_adv[rookie_adv['season_start'] == rookie_adv['DraftYear']+1]

rookie_avg = avg.merge(players_sel, on="Player", how="inner")
rookie_avg['season_start'] = rookie_avg['season'].str[:4].astype(int)
rookie_avg = rookie_avg[rookie_avg['season_start'] == rookie_avg['DraftYear']+1]

# 合併菜鳥年數據
rookie_stats = pd.merge(
    rookie_avg,
    rookie_adv,
    on=["Player","season","team"],
    suffixes=("_basic","_adv")
)

# 整合最終表格
final = players_sel.merge(career_adv, on="Player", how="left").merge(rookie_stats, on="Player", how="left")

# 輸出
final.to_excel("NBA_Stats_1995_2015.xlsx", index=False)
print("✅ 已輸出 NBA_Stats_1995_2015.xlsx")


In [ ]:
import pandas as pd

# 讀檔案
adv = pd.read_parquet("advanced.parq")
avg = pd.read_parquet("average.parq")
roster = pd.read_parquet("roster.parq")

# season 格式是 "1996/1997"，先取前4碼轉成 int
roster['season_start'] = roster['season'].str[:4].astype(int)

# 推算 DraftYear = 最早出現的賽季 - 1
roster['DraftYear'] = roster.groupby("Player")['season_start'].transform('min') - 1

# 篩選 1995–2015 DraftYear，保留位置 Pos
players_sel = roster[['Player','DraftYear','Pos']].drop_duplicates()
players_sel = players_sel[(players_sel['DraftYear'] >= 1995) & (players_sel['DraftYear'] <= 2015)]

# 生涯數據
career_adv = adv.groupby("Player").agg({
    "WS": "sum",
    "PER": "mean",
    "BPM": "mean"
}).reset_index().rename(columns={"WS":"Career_WS","PER":"Career_PER","BPM":"Career_BPM"})

# 菜鳥年 (DraftYear+1)
rookie_adv = adv.merge(players_sel, on="Player", how="inner")
rookie_adv['season_start'] = rookie_adv['season'].str[:4].astype(int)
rookie_adv = rookie_adv[rookie_adv['season_start'] == rookie_adv['DraftYear']+1]

rookie_avg = avg.merge(players_sel, on="Player", how="inner")
rookie_avg['season_start'] = rookie_avg['season'].str[:4].astype(int)
rookie_avg = rookie_avg[rookie_avg['season_start'] == rookie_avg['DraftYear']+1]

# 合併菜鳥年數據
rookie_stats = pd.merge(
    rookie_avg,
    rookie_adv,
    on=["Player","season","team"],
    suffixes=("_basic","_adv")
)

# 整合最終表格 (含 Pos)
final = players_sel.merge(career_adv, on="Player", how="left").merge(rookie_stats, on="Player", how="left")

# 輸出
final.to_excel("NBA_Stats_1995_2015.xlsx", index=False)
print("✅ 已輸出 NBA_Stats_1995_2015.xlsx，包含球員位置 Pos")


In [ ]:
import pandas as pd

# 讀檔案
adv = pd.read_parquet("advanced.parq")
avg = pd.read_parquet("average.parq")
roster = pd.read_parquet("roster.parq")

# season 格式是 "1996/1997"，先取前4碼轉成 int
roster['season_start'] = roster['season'].str[:4].astype(int)

# 推算 DraftYear = 最早出現的賽季 - 1
roster['DraftYear'] = roster.groupby("Player")['season_start'].transform('min') - 1

# 篩選 1995–2015 DraftYear，保留位置 Pos
players_sel = roster[['Player','DraftYear','Pos']].drop_duplicates()
players_sel = players_sel[(players_sel['DraftYear'] >= 1990) & (players_sel['DraftYear'] <= 2015)]

# 生涯數據
career_adv = adv.groupby("Player").agg({
    "WS": "sum",
    "PER": "mean",
    "BPM": "mean"
}).reset_index().rename(columns={"WS":"Career_WS","PER":"Career_PER","BPM":"Career_BPM"})

# 菜鳥年 (DraftYear+1)
rookie_adv = adv.merge(players_sel, on="Player", how="inner")
rookie_adv['season_start'] = rookie_adv['season'].str[:4].astype(int)
rookie_adv = rookie_adv[rookie_adv['season_start'] == rookie_adv['DraftYear']+1]

rookie_avg = avg.merge(players_sel, on="Player", how="inner")
rookie_avg['season_start'] = rookie_avg['season'].str[:4].astype(int)
rookie_avg = rookie_avg[rookie_avg['season_start'] == rookie_avg['DraftYear']+1]

# 合併菜鳥年數據
rookie_stats = pd.merge(
    rookie_avg,
    rookie_adv,
    on=["Player","season","team"],
    suffixes=("_basic","_adv")
)

# 整合最終表格 (含 Pos)
final = players_sel.merge(career_adv, on="Player", how="left").merge(rookie_stats, on="Player", how="left")

final = final.drop_duplicates(subset=['Player'], keep='first')
# 輸出
final.to_excel("NBA_Stats_1995_2015.xlsx", index=False)
print("✅ 已輸出 NBA_Stats_1995_2015.xlsx，包含球員位置 Pos")


In [ ]:
import pandas as pd

# 讀取檔案
df = pd.read_excel("NBA_Stats_1995_2015.xlsx")

# 假設你原始有一個「每季的 advanced 數據」DataFrame adv
# adv 需要至少包含: Player, season, WS
adv = pd.read_parquet("advanced.parq")  # 換成你的檔案路徑

# 計算每位球員的出賽賽季數
career_seasons = adv.groupby("Player")["season"].nunique().reset_index()
career_seasons.rename(columns={"season": "Career_Seasons"}, inplace=True)

# 合併進去
df = df.merge(career_seasons, on="Player", how="left")

# 計算平均 WS (累積 WS ÷ 賽季數)
df["Career_WS_avg"] = df["Career_WS"] / df["Career_Seasons"]

# 存回原檔
df.to_excel("NBA_Stats_1995_2015.xlsx", index=False)

print("✅ 已經在檔案裡新增 Career_WS_avg 欄位")


In [ ]:
import pandas as pd

# 讀取已整理好的檔案
df = pd.read_excel("NBA_Stats_1995_2015.xlsx")

# 讀取 Kaggle 的 roster 或 average，這裡用 average (因為它有 season 資訊)
average = pd.read_parquet("average.parq")

# 每位球員打過幾個不同賽季
season_counts = average.groupby("Player")['season'].nunique().reset_index()
season_counts = season_counts.rename(columns={'season': 'season_played'})

# 合併回原始 df
df = df.merge(season_counts, on="Player", how="left")

# 存回檔案
df.to_excel("NBA_Stats_1995_2015.xlsx", index=False)

print("✅ 已經加上 season_played 欄位")


In [11]:
import pandas as pd
import numpy as np

# 讀取 CSV
df = pd.read_csv("Seasons_Stats.csv")

# 清理欄位
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.columns = df.columns.str.strip()
df["Player"] = df["Player"].astype(str).str.strip()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# 找各球員首次出現（視為進入NBA）
first_years = (
    df.groupby("Player", as_index=False)["Year"]
      .min()
      .rename(columns={"Year": "First_Year"})
)

# 篩 1990–2010 首度進入NBA的球員
target = first_years[(first_years["First_Year"] >= 1980) & (first_years["First_Year"] <= 2010)]

# 合併並取前兩年資料
merged = df.merge(target, on="Player", how="inner")
merged = merged[merged["Year"].between(merged["First_Year"], merged["First_Year"] + 2)]

# 去掉全空欄
merged = merged.dropna(axis=1, how="all")

# 分出數值與文字欄
num_cols = merged.select_dtypes(include="number").columns.tolist()
# 不要對 Year 取平均，First_Year 只是鍵值也不平均
for c in ["Year", "First_Year"]:
    if c in num_cols:
        num_cols.remove(c)

cat_cols = merged.select_dtypes(exclude="number").columns.tolist()
# Player 是鍵，不需要取眾數
if "Player" in cat_cols:
    cat_cols.remove("Player")

# 數值欄位：前兩年平均
num_avg = (
    merged.groupby(["Player", "First_Year"], as_index=False)[num_cols]
          .mean()
)

# 文字欄位：取眾數（無眾數則回傳 NA）
def mode_or_na(x):
    m = x.mode(dropna=True)
    return m.iloc[0] if not m.empty else pd.NA

if cat_cols:
    cat_mode = (
        merged.groupby(["Player", "First_Year"], as_index=False)[cat_cols]
              .agg(mode_or_na)
    )
    final = num_avg.merge(cat_mode, on=["Player", "First_Year"], how="left")
else:
    final = num_avg.copy()

# 排序與輸出
final = final.sort_values(["First_Year", "Player"], kind="mergesort").reset_index(drop=True)
final.to_csv("NBA_Normalized_Final_3.csv", index=False)
print(f"✅ 已輸出：NBA_First2Years_Avg_Full_1990_2010.xlsx")
print(f"🧾 球員數：{final.shape[0]}，欄位數：{final.shape[1]}")


✅ 已輸出：NBA_First2Years_Avg_Full_1990_2010.xlsx
🧾 球員數：2046，欄位數：50


In [8]:
import pandas as pd
import numpy as np

# 讀取資料
df = pd.read_csv("Seasons_Stats.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.columns = df.columns.str.strip()
df["Player"] = df["Player"].astype(str).str.strip()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# 定義累計欄位（需要除以 G）
total_cols = [
    "FG", "FGA", "3P", "3PA", "2P", "2PA", "FT", "FTA",
    "ORB", "DRB", "TRB", "AST", "STL", "BLK", "TOV", "PF", "PTS"
]

# 先過濾掉沒有出賽場次的行
df = df[df["G"].notna() & (df["G"] > 0)]

# 對累計欄位除以 G，轉為 per-game
for col in total_cols:
    if col in df.columns:
        df[col] = df[col] / df["G"]

# 找出每位球員的首個賽季
first_years = (
    df.groupby("Player", as_index=False)["Year"]
      .min()
      .rename(columns={"Year": "First_Year"})
)

# 篩選 1990–2010 首次進入 NBA 的球員
target = first_years[(first_years["First_Year"] >= 1990) & (first_years["First_Year"] <= 2010)]

# 合併後取前兩年資料
merged = df.merge(target, on="Player", how="inner")
merged = merged[merged["Year"].between(merged["First_Year"], merged["First_Year"] + 1)]

# 移除全空欄位
merged = merged.dropna(axis=1, how="all")

# 分數值與文字欄
num_cols = merged.select_dtypes(include="number").columns.tolist()
for c in ["Year", "First_Year"]:
    if c in num_cols:
        num_cols.remove(c)

cat_cols = merged.select_dtypes(exclude="number").columns.tolist()
if "Player" in cat_cols:
    cat_cols.remove("Player")

# 數值欄取平均
num_avg = (
    merged.groupby(["Player", "First_Year"], as_index=False)[num_cols]
          .mean()
)

# 文字欄取眾數
def mode_or_na(x):
    m = x.mode(dropna=True)
    return m.iloc[0] if not m.empty else pd.NA

if cat_cols:
    cat_mode = (
        merged.groupby(["Player", "First_Year"], as_index=False)[cat_cols]
              .agg(mode_or_na)
    )
    final = num_avg.merge(cat_mode, on=["Player", "First_Year"], how="left")
else:
    final = num_avg.copy()

# 排序與輸出
final = final.sort_values(["First_Year", "Player"], kind="mergesort").reset_index(drop=True)
final.to_excel("NBA_First2Years_Avg_PerGame_1990_2010.xlsx", index=False)
print(f"✅ 已輸出：NBA_First2Years_Avg_PerGame_1990_2010.xlsx")
print(f"🧾 球員數：{final.shape[0]}，欄位數：{final.shape[1]}")


✅ 已輸出：NBA_First2Years_Avg_PerGame_1990_2010.xlsx
🧾 球員數：1418，欄位數：50


In [7]:
import pandas as pd

# 1️⃣ 讀取兩份檔案
career_df = pd.read_csv("player_data.csv")
avg_df = pd.read_csv("NBA_Normalized_Final_3.csv")

# 2️⃣ 清理欄位名稱與格式
career_df.columns = career_df.columns.str.strip()
avg_df.columns = avg_df.columns.str.strip()

career_df["name"] = career_df["name"].astype(str).str.strip()
avg_df["Player"] = avg_df["Player"].astype(str).str.strip()

# 3️⃣ 計算每位球員的生涯年數 (Year_end - Year_start + 1)
career_df["season"] = career_df["year_end"] - career_df["year_start"] + 1

# 4️⃣ 只保留 name 與 season
career_seasons = career_df[["name", "season"]].copy()

# 5️⃣ 合併 — 將 season 加進原檔
merged = avg_df.merge(career_seasons, left_on="Player", right_on="name", how="left")

# 6️⃣ 清理（移除多餘的 name 欄位）
merged = merged.drop(columns=["name"], errors="ignore")

# 7️⃣ 輸出
output_file = "NBA_Normalized_Final_3.csv"
merged.to_excel(output_file, index=False)

print(f"✅ 已輸出：{output_file}")
print(f"📘 共 {merged.shape[0]} 位球員，欄位數 {merged.shape[1]}。")


FileNotFoundError: [Errno 2] No such file or directory: 'player_data.csv'

In [11]:
import pandas as pd

# 1️⃣ 讀取兩個檔案
base_df = pd.read_excel("NBA_First2Years_Avg_PerGame_1990_2010_withSeason.xlsx")
adv_df = pd.read_excel("merged_output (4).xlsx")

# 2️⃣ 清理欄位名稱與格式
base_df.columns = base_df.columns.str.strip()
adv_df.columns = adv_df.columns.str.strip()

base_df["Player"] = base_df["Player"].astype(str).str.strip()
adv_df["Player"] = adv_df["Player"].astype(str).str.strip()

# 3️⃣ 篩選需要的進階欄位（若部分不存在也不會報錯）
adv_cols = ["Player", "Advanced_WS", "Advanced_WS/48", "Advanced_BPM", "Advanced_VORP"]
adv_existing = [c for c in adv_cols if c in adv_df.columns]

adv_selected = adv_df[adv_existing].copy()

# 4️⃣ 合併（依球員名稱）
merged = base_df.merge(adv_selected, on="Player", how="left")

# 5️⃣ 輸出
output_file = "NBA_First2Years_Avg_PerGame_1990_2010_withSeason_Advanced.xlsx"
merged.to_excel(output_file, index=False)

print(f"✅ 已輸出：{output_file}")
print(f"📘 共 {merged.shape[0]} 位球員，欄位數 {merged.shape[1]}。")


✅ 已輸出：NBA_First2Years_Avg_PerGame_1990_2010_withSeason_Advanced.xlsx
📘 共 1433 位球員，欄位數 55。


In [12]:
import pandas as pd

# 讀取原始檔案
df = pd.read_excel("NBA_First2Years_Avg_PerGame_1990_2010_withSeason_Advanced.xlsx")

# 清理欄位
df.columns = df.columns.str.strip()
df["Player"] = df["Player"].astype(str).str.strip()

# 拆分 Player 成 first_name / last_name
df["first_name"] = df["Player"].apply(lambda x: x.split(" ")[0] if isinstance(x, str) else "")
df["last_name"] = df["Player"].apply(lambda x: " ".join(x.split(" ")[1:]) if isinstance(x, str) and len(x.split(" ")) > 1 else "")

# 調整欄位順序：讓 first_name 和 last_name 緊跟在 Player 後面
cols = list(df.columns)
insert_at = cols.index("Player") + 1
new_order = cols[:insert_at] + ["first_name", "last_name"] + cols[insert_at:]
df = df[new_order]

# 輸出結果
output_file = "NBA_First2Years_with_FirstLastName.xlsx"
df.to_excel(output_file, index=False)

print(f"✅ 已完成拆分，輸出檔案：{output_file}")
print(f"📘 新增欄位：first_name, last_name，共 {df.shape[0]} 筆球員資料。")


✅ 已完成拆分，輸出檔案：NBA_First2Years_with_FirstLastName.xlsx
📘 新增欄位：first_name, last_name，共 1430 筆球員資料。


In [14]:
import pandas as pd

# 1️⃣ 讀取資料
season_df = pd.read_csv("Seasons_Stats.csv")
main_df = pd.read_excel("NBA_First2Years_with_FirstLastName.xlsx")

# 2️⃣ 清理欄位
season_df.columns = season_df.columns.str.strip()
main_df.columns = main_df.columns.str.strip()
season_df["Player"] = season_df["Player"].astype(str).str.strip()
main_df["Player"] = main_df["Player"].astype(str).str.strip()

# 3️⃣ 篩選 DraftYear <= 2006
if "DraftYear" in main_df.columns:
    main_df = main_df[main_df["First_Year"] <= 2006].copy()

# 4️⃣ 計算生涯進階數據
career_stats = season_df.groupby("Player").agg({
    "WS": "sum",       # WS 要加總
    "WS/48": "mean",   # WS/48 平均
    "BPM": "mean",     # BPM 平均
    "VORP": "mean"     # VORP 平均
}).reset_index()

career_stats.rename(columns={
    "WS": "Advanced_WS",
    "WS/48": "Advanced_WS/48",
    "BPM": "Advanced_BPM",
    "VORP": "Advanced_VORP"
}, inplace=True)

# 5️⃣ 合併（補缺值）
merged = main_df.merge(career_stats, on="Player", how="left", suffixes=("", "_new"))

for col in ["Advanced_WS", "Advanced_WS/48", "Advanced_BPM", "Advanced_VORP"]:
    new_col = f"{col}_new"
    if col not in merged.columns:
        merged.rename(columns={new_col: col}, inplace=True)
    elif new_col in merged.columns:
        merged[col] = merged[col].fillna(merged[new_col])
        merged.drop(columns=[new_col], inplace=True, errors="ignore")

# 6️⃣ 匯出
output_file = "NBA_First2Years_with_FirstLastName_AdvancedFilled.xlsx"
merged.to_excel(output_file, index=False)

print(f"✅ 已成功補齊 Advanced 欄位數據！")
print(f"📘 輸出檔案：{output_file}")
print(f"📊 共 {merged.shape[0]} 位球員，欄位數 {merged.shape[1]}。")


✅ 已成功補齊 Advanced 欄位數據！
📘 輸出檔案：NBA_First2Years_with_FirstLastName_AdvancedFilled.xlsx
📊 共 1426 位球員，欄位數 57。


In [1]:
import pandas as pd
import requests
import time
import random
from io import StringIO

def get_player_slug(name):
    """將姓名轉換為 Basketball Reference ID (例如: Michael Redd -> reddmi01)"""
    clean_name = name.replace("'", "").replace(".", "").replace("-", " ").lower().split()
    if len(clean_name) < 2: return None
    last = clean_name[-1]
    first = clean_name[0]
    slug = f"{last[:5]}{first[:2]}01"
    return slug

def fetch_advanced_stats(player_name):
    slug = get_player_slug(player_name)
    if not slug: return None
    
    url = f"https://www.basketball-reference.com/players/{slug[0]}/{slug}.html"
    headers = {'User-Agent': 'Mozilla/5.0'}
    
    print(f"正在分析 Advanced 表格: {player_name}...")
    
    try:
        # 隨機延遲 4-7 秒，避免被封鎖
        time.sleep(random.uniform(4, 7))
        
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"  [跳過] 無法訪問頁面")
            return None

        # 讀取網頁中的所有表格
        # 使用 StringIO 包裝 response.text 以避免 FutureWarning
        tables = pd.read_html(StringIO(response.text))
        
        adv_df = None
        for t in tables:
            # 辨識 Advanced 統計表
            if 'WS' in t.columns and 'WS/48' in t.columns and 'Season' in t.columns:
                adv_df = t
                break
        
        if adv_df is not None:
            # --- 重要：清洗表格數據 ---
            # 1. 只保留包含球季格式 (如 2000-01) 的行
            adv_df = adv_df[adv_df['Season'].str.contains("-", na=False)].copy()
            
            # 2. 處理「季中轉隊」：如果同一個 Season 出現多次，只保留第一行 (通常是該季總計 'TOT')
            adv_df = adv_df.drop_duplicates(subset=['Season'], keep='first').reset_index(drop=True)
            
            res = {}
            # 索引 2 代表第 3 個賽季，索引 3 代表第 4 個賽季
            try:
                res['Year3_Actual_WS'] = float(adv_df.iloc[2]['WS'])
                res['Year3_Actual_WS48'] = float(adv_df.iloc[2]['WS/48'])
            except:
                res['Year3_Actual_WS'] = None
                res['Year3_Actual_WS48'] = None
                
            try:
                res['Year4_Actual_WS'] = float(adv_df.iloc[3]['WS'])
                res['Year4_Actual_WS48'] = float(adv_df.iloc[3]['WS/48'])
            except:
                res['Year4_Actual_WS'] = None
                res['Year4_Actual_WS48'] = None
                
            return res
            
    except Exception as e:
        print(f"  [錯誤] {player_name}: {e}")
        return None

# --- 主執行程序 ---
file_name = "NBA_Normalized_Final.csv"

try:
    df = pd.read_csv(file_name)
    print(f"讀取成功，準備更新 {len(df)} 筆資料。")
except FileNotFoundError:
    print(f"找不到檔案: {file_name}")
    exit()

# 確保欄位存在
for col in ['Year3_Actual_WS', 'Year3_Actual_WS48', 'Year4_Actual_WS', 'Year4_Actual_WS48']:
    if col not in df.columns:
        df[col] = None

# 執行抓取
for index, row in df.iterrows():
    # 如果已經抓過數據，可以選擇跳過 (加速處理)
    if pd.notnull(df.at[index, 'Year3_Actual_WS']):
        continue
        
    stats = fetch_advanced_stats(row['Player'])
    if stats:
        for key, value in stats.items():
            df.at[index, key] = value
    
    # 每 5 筆強制存檔一次，確保資料安全
    if index % 5 == 0:
        df.to_csv(file_name, index=False)

# 最終存檔
df.to_csv(file_name, index=False)
print(f"數據已成功覆寫至 {file_name}")

C:\Users\sara9\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


讀取成功，準備更新 2059 筆資料。
正在分析 Advanced 表格: David Robinson*...
  [跳過] 無法訪問頁面
正在分析 Advanced 表格: Jeff Hornacek...
  [跳過] 無法訪問頁面
正在分析 Advanced 表格: Bill Laimbeer...


KeyboardInterrupt: 

In [ ]:
import pandas as pd
import numpy as np

# 讀取資料
df = pd.read_csv("Seasons_Stats.csv")
df = df.drop(columns=["Unnamed: 0"], errors="ignore")
df.columns = df.columns.str.strip()
df["Player"] = df["Player"].astype(str).str.strip()
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")

# 定義累計欄位（需要除以 G）
total_cols = [
    "FG", "FGA", "3P", "3PA", "2P", "2PA", "FT", "FTA",
    "ORB", "DRB", "TRB", "AST", "STL", "BLK", "TOV", "PF", "PTS"
]

# 先過濾掉沒有出賽場次的行
df = df[df["G"].notna() & (df["G"] > 0)]

# 對累計欄位除以 G，轉為 per-game
for col in total_cols:
    if col in df.columns:
        df[col] = df[col] / df["G"]

# 找出每位球員的首個賽季
first_years = (
    df.groupby("Player", as_index=False)["Year"]
      .min()
      .rename(columns={"Year": "First_Year"})
)

# 篩選 1990–2010 首次進入 NBA 的球員
target = first_years[(first_years["First_Year"] >= 1990) & (first_years["First_Year"] <= 2010)]

# 合併後取前兩年資料
merged = df.merge(target, on="Player", how="inner")
merged = merged[merged["Year"].between(merged["First_Year"], merged["First_Year"] + 1)]

# 移除全空欄位
merged = merged.dropna(axis=1, how="all")

# 分數值與文字欄
num_cols = merged.select_dtypes(include="number").columns.tolist()
for c in ["Year", "First_Year"]:
    if c in num_cols:
        num_cols.remove(c)

cat_cols = merged.select_dtypes(exclude="number").columns.tolist()
if "Player" in cat_cols:
    cat_cols.remove("Player")

# 數值欄取平均
num_avg = (
    merged.groupby(["Player", "First_Year"], as_index=False)[num_cols]
          .mean()
)

# 文字欄取眾數
def mode_or_na(x):
    m = x.mode(dropna=True)
    return m.iloc[0] if not m.empty else pd.NA

if cat_cols:
    cat_mode = (
        merged.groupby(["Player", "First_Year"], as_index=False)[cat_cols]
              .agg(mode_or_na)
    )
    final = num_avg.merge(cat_mode, on=["Player", "First_Year"], how="left")
else:
    final = num_avg.copy()

# 排序與輸出
final = final.sort_values(["First_Year", "Player"], kind="mergesort").reset_index(drop=True)
final.to_excel("NBA_First2Years_Avg_PerGame_1990_2010.xlsx", index=False)
print(f"✅ 已輸出：NBA_First2Years_Avg_PerGame_1990_2010.xlsx")
print(f"🧾 球員數：{final.shape[0]}，欄位數：{final.shape[1]}")


In [3]:
import pandas as pd

# 1. 設定檔案名稱
main_file = "NBA_Normalized_Final.csv"  # 你的主檔案 (含 Player, Year_Start)
stats_file = "Seasons_Stats.csv"    # 包含球員每一年資料的另一個檔案

try:
    # 2. 讀取檔案
    df_main = pd.read_csv(main_file)
    df_all = pd.read_csv(stats_file)
    
    print(f"主檔案讀取成功：{len(df_main)} 位球員")
    print(f"歷年資料讀取成功：{len(df_all)} 筆紀錄")

    # 3. 確保年份欄位是數字格式
    # 假設主檔案的起始年欄位叫 'Year_Start'，歷年資料的年份欄位叫 'Season_Year'
    # 請根據你實際的欄位名稱修改下面這兩個字串
    start_col = 'year_start' 
    data_year_col = 'Year'

    # 4. 定義抓取邏輯
    results = []

    for index, row in df_main.iterrows():
        player = row['Player']
        rookie_year = row[start_col]
        
        # 計算第三年與第四年的目標年份
        year_3_target = rookie_year + 2
        year_4_target = rookie_year + 3
        
        # 在歷年資料中尋找該球員對應年份的資料
        # 篩選條件：名字相同 且 年份符合
        y3_data = df_all[(df_all['Player'] == player) & (df_all[data_year_col] == year_3_target)]
        y4_data = df_all[(df_all['Player'] == player) & (df_all[data_year_col] == year_4_target)]
        
        # 提取 WS 與 WS/48 (假設欄位名為 'WS' 和 'WS/48')
        y3_ws = y3_data['WS'].values[0] if not y3_data.empty else None
        y3_ws48 = y3_data['WS/48'].values[0] if not y3_data.empty else None
        y4_ws = y4_data['WS'].values[0] if not y4_data.empty else None
        y4_ws48 = y4_data['WS/48'].values[0] if not y4_data.empty else None
        
        # 更新主表資料
        df_main.at[index, 'Year3_Actual_WS'] = y3_ws
        df_main.at[index, 'Year3_Actual_WS48'] = y3_ws48
        df_main.at[index, 'Year4_Actual_WS'] = y4_ws
        df_main.at[index, 'Year4_Actual_WS48'] = y4_ws48

    # 5. 直接覆寫回原檔案
    df_main.to_csv(main_file, index=False)
    print(f"合併完成！數據已更新並覆寫至 {main_file}")

except Exception as e:
    print(f"處理出錯: {e}")

主檔案讀取成功：2059 位球員
歷年資料讀取成功：24691 筆紀錄
合併完成！數據已更新並覆寫至 NBA_Normalized_Final.csv


In [11]:
import pandas as pd

def get_market_context(year):
    """
    根據年份回傳當年的 Salary Cap (百萬美金) 與建議的 BRI Factor
    """
    # 歷年 Salary Cap 數據 (單位: 百萬美元)
    caps = {
        1984: 3.6, 1985: 4.2, 1986: 4.9, 1987: 6.2, 1988: 7.2, 1989: 9.8,
        1990: 11.9, 1991: 12.5, 1992: 14.0, 1993: 15.2, 1994: 16.0, 1995: 23.0,
        1996: 24.4, 1997: 26.9, 1998: 30.0, 1999: 34.0, 2000: 35.5, 2001: 42.5,
        2002: 40.3, 2003: 43.8, 2004: 43.9, 2005: 49.5, 2006: 53.1, 2007: 55.6,
        2008: 58.7, 2009: 57.7, 2010: 58.0
    }
    
    # 根據 CBA 協議區間設定 BRI Factor
    # 1999 與 2005 為關鍵的分水嶺年份
    if year < 1999:
        bri = 1.10
    elif 1999 <= year <= 2004:
        bri = 1.18
    elif 2005 <= year <= 2010:
        bri = 1.25
    else:
        bri = 1.20 # 預設值
        
    return caps.get(year, 30.0), bri

# 1. 讀取檔案
file_name = "NBA_Normalized_Final.csv"
try:
    df = pd.read_csv(file_name)
    print(f"成功讀取 {file_name}，共 {len(df)} 筆資料。")
except FileNotFoundError:
    print(f"找不到檔案 {file_name}，請確認檔案在正確目錄下。")
    exit()

# 2. 刪除 Year_Start 是空白的行
initial_count = len(df)
df = df.dropna(subset=['year_start'])
deleted_count = initial_count - len(df)
if deleted_count > 0:
    print(f"已刪除 {deleted_count} 列 Year_Start 為空白的無效資料。")

# 3. 加入 Salary Cap 與 BRI Factor
# 確保年份為整數格式以便對應
df['year_start'] = df['year_start'].astype(int)

# 透過 apply 一次抓取兩個值
df[['Salary_Cap_At_Start', 'BRI_Factor']] = df['year_start'].apply(
    lambda y: pd.Series(get_market_context(y))
)

# 4. 直接覆寫回 CSV 檔案
df.to_csv(file_name, index=False)
print(f"處理完成！已更新 Salary_Cap_At_Start 與 BRI_Factor 並覆寫至 {file_name}")

# 印出前 5 筆檢查
print(df[['Player', 'year_start', 'Salary_Cap_At_Start', 'BRI_Factor']].head())

成功讀取 NBA_Normalized_Final.csv，共 2011 筆資料。
處理完成！已更新 Salary_Cap_At_Start 與 BRI_Factor 並覆寫至 NBA_Normalized_Final.csv
            Player  year_start  Salary_Cap_At_Start  BRI_Factor
0  David Robinson*        1990                 11.9        1.10
1    Jeff Hornacek        1987                  6.2        1.10
2    Bill Laimbeer        1981                 30.0        1.10
3    Manu Ginobili        2002                 40.3        1.18
4   DeAndre Jordan        2009                 57.7        1.25


In [58]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. 讀取原始檔案
# ==========================================
df_seasons = pd.read_csv('Seasons_Stats.csv')
df_normalized = pd.read_csv('NBA_Normalized_Final.csv')

# ==========================================
# 2. 名字標準化函數
# ==========================================
def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = re.sub(r'[^a-z]', '', name)
    for suf in ["jr", "sr", "ii", "iii", "iv"]:
        if name.endswith(suf):
            name = name[:-len(suf)]
    return name

# ==========================================
# 3. 找出每位球員進聯盟第一年
# ==========================================
df_seasons = df_seasons.dropna(subset=['Player', 'Year']).copy()
df_seasons['Year'] = pd.to_numeric(df_seasons['Year'], errors='coerce')
df_seasons = df_seasons.dropna(subset=['Year']).copy()
df_seasons['Year'] = df_seasons['Year'].astype(int)

player_start = df_seasons.groupby('Player', as_index=False)['Year'].min()
player_start.columns = ['Player', 'year_start']

# 只抓 2011 ~ 2016 進聯盟球員
target_players = player_start[
    (player_start['year_start'] >= 2011) &
    (player_start['year_start'] <= 2016)
].copy()

# 只保留目標球員原始資料
df_target_raw = df_seasons[df_seasons['Player'].isin(target_players['Player'])].copy()
df_target_raw = df_target_raw.merge(target_players, on='Player', how='inner')

# ==========================================
# 4. 每季資料：取消 TOT 優先，直接取第一筆
# ==========================================
def get_season_data(df_group):
    return df_group.iloc[0]

# ==========================================
# 5. 重新建立 2011~2016 球員資料
# ==========================================
processed_rows = []

for player, group in df_target_raw.groupby('Player'):
    start_y = int(group['year_start'].iloc[0])

    # 每季只留一筆（不再 TOT 優先）
    year_map = {
        int(year): get_season_data(year_group)
        for year, year_group in group.groupby('Year')
    }

    # 前兩年
    y1 = year_map.get(start_y)
    y2 = year_map.get(start_y + 1)
    active_seasons = [row for row in [y1, y2] if row is not None]

    if len(active_seasons) == 0:
        continue

    # 計算前兩年平均（只對數值欄位）
    avg_stats = pd.DataFrame(active_seasons).mean(numeric_only=True).to_dict()

    # 保留基本欄位
    avg_stats['Player'] = player
    avg_stats['Pos'] = y1['Pos'] if y1 is not None and 'Pos' in y1 else np.nan
    avg_stats['Tm'] = y1['Tm'] if y1 is not None and 'Tm' in y1 else np.nan
    avg_stats['year_start'] = start_y
    avg_stats['season'] = len(active_seasons)

    # 第三、四年目標值
    y3 = year_map.get(start_y + 2)
    y4 = year_map.get(start_y + 3)
    avg_stats['Year3_WS'] = y3['WS'] if y3 is not None and 'WS' in y3 else np.nan
    avg_stats['Year4_WS'] = y4['WS'] if y4 is not None and 'WS' in y4 else np.nan

    processed_rows.append(avg_stats)

df_new = pd.DataFrame(processed_rows)

# ==========================================
# 6. 建立位置 dummy 欄位
# ==========================================
for pos in ['C', 'PF', 'PG', 'SF', 'SG']:
    df_new[f'Pos_{pos}'] = df_new['Pos'].apply(
        lambda x: 1 if pd.notna(x) and pos in str(x) else 0
    )

# ==========================================
# 7. 建立乾淨名字鍵
# ==========================================
df_new['clean_name'] = df_new['Player'].apply(clean_name)
df_normalized['clean_name'] = df_normalized['Player'].apply(clean_name)

# ==========================================
# 8. 排除生涯總數據欄位
# ==========================================
exclude_advanced = ['Advanced_VORP', 'Advanced_BPM', 'Advanced_WS', 'Advanced_WS/48']

final_cols = df_normalized.columns.tolist()

for col in final_cols:
    if col not in df_new.columns or col in exclude_advanced:
        df_new[col] = np.nan

df_new = df_new[final_cols].copy()

# ==========================================
# 9. 先刪舊資料，再加入新資料
# ==========================================
target_clean_names = set(df_new['clean_name'].dropna().unique())

df_old_kept = df_normalized[
    ~df_normalized['clean_name'].isin(target_clean_names)
].copy()

df_updated = pd.concat([df_old_kept, df_new], ignore_index=True)

# 去重
df_updated = df_updated.drop_duplicates(
    subset=['clean_name'],
    keep='last'
).copy()

# ==========================================
# 10. 清理輔助欄位並存檔
# ==========================================
if 'clean_name' in df_updated.columns:
    df_updated = df_updated.drop(columns=['clean_name'])

df_updated.to_csv(
    'NBA_Normalized_Final.csv',
    index=False,
    encoding='utf-8-sig'
)

print(f"處理完成！已重新抓取並覆蓋 {len(df_new)} 位 2011-2016 起始球員。")
print(f"目前總筆數：{len(df_updated)}")

處理完成！已重新抓取並覆蓋 448 位 2011-2016 起始球員。
目前總筆數：2648


In [ ]:
import pandas as pd
import numpy as np
import time
import random
import re

# 偽裝瀏覽器標頭
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def fetch_nba_data(year):
    """抓取特定年份的每場數據與進階數據"""
    print(f"--- 正在抓取 {year} 年數據 ---")
    
    # 1. 抓取 Per Game (基礎數據)
    url_pg = f"https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html"
    df_pg = pd.read_html(url_pg)[0]
    df_pg = df_pg[df_pg['Player'] != 'Player'].copy()
    
    # 隨機休息
    wait_time = random.uniform(15, 25)
    print(f"等待 {wait_time:.2f} 秒...")
    time.sleep(wait_time)
    
    # 2. 抓取 Advanced (進階數據)
    url_adv = f"https://www.basketball-reference.com/leagues/NBA_{year}_advanced.html"
    df_adv = pd.read_html(url_adv)[0]
    df_adv = df_adv[df_adv['Player'] != 'Player'].copy()
    
    # 合併兩者
    df_year = pd.merge(df_pg, df_adv[['Player', 'Tm', 'PER', 'TS%', 'WS', 'WS/48', 'BPM', 'VORP', 'USG%']], on=['Player', 'Tm'], how='left')
    df_year['Year'] = year
    return df_year

# ---------------------------------------------------------
# 第一步：執行抓取 (建議分批執行，這裡示範 2017-2022)
# ---------------------------------------------------------
all_years_data = []
for y in range(2017, 2023):
    try:
        data = fetch_nba_data(y)
        all_years_data.append(data)
    except Exception as e:
        print(f"{y} 年發生錯誤: {e}")

df_raw = pd.concat(all_years_data, ignore_index=True)

# ---------------------------------------------------------
# 第二步：資料清洗與平均值計算
# ---------------------------------------------------------
# 轉換數值
numeric_cols = df_raw.columns.drop(['Player', 'Pos', 'Tm'])
for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# 找出每位球員的第一年
player_start = df_raw.groupby('Player')['Year'].min().reset_index()
player_start.columns = ['Player', 'year_start']

# 篩選 2017-2022 開始打球的球員
target_players = player_start[player_start['year_start'] >= 2017]
df_target = df_raw[df_raw['Player'].isin(target_players['Player'])].copy()
df_target = df_target.merge(target_players, on='Player')

processed_data = []
for player, group in df_target.groupby('Player'):
    start_y = group['year_start'].iloc[0]
    
    # 處理 TOT (交易)
    def clean_trades(g):
        if len(g) > 1:
            tot = g[g['Tm'] == 'TOT']
            return tot.iloc[0] if not tot.empty else g.iloc[0]
        return g.iloc[0]

    y_map = {int(y): clean_trades(y_g) for y, y_g in group.groupby('Year')}
    
    # 前兩年平均 (Basic Stats)
    seasons = [y_map.get(int(start_y)), y_map.get(int(start_y + 1))]
    active = [s for s in seasons if s is not None]
    
    if not active: continue
    
    # A. 計算前兩年平均
    row = pd.DataFrame(active).mean(numeric_only=True).to_dict()
    row['Player'] = player
    row['year_start'] = start_y
    row['Pos'] = active[0]['Pos']
    
    # B. 抓取生涯總數據 (這就是你要求的那四個欄位)
    # 這裡用 group 的 sum 或平均來代表目前為止的生涯狀態
    row['Advanced_WS'] = group['WS'].sum()
    row['Advanced_BPM'] = group['BPM'].mean()
    row['Advanced_VORP'] = group['VORP'].sum()
    row['Advanced_WS/48'] = group['WS/48'].mean()
    
    processed_data.append(row)

df_final_new = pd.DataFrame(processed_data)


In [62]:
import pandas as pd
import numpy as np
import re

# =====================================================
# 1. 讀取資料
# =====================================================
SEASON_FILE = "Seasons_Stats.csv"
MAIN_FILE = "NBA_Normalized_Final.csv"
OUTPUT_FILE = "NBA_Normalized_Final.csv"

df_season = pd.read_csv(SEASON_FILE)
df_main = pd.read_csv(MAIN_FILE)

# =====================================================
# 2. 名字清理函數
# =====================================================
def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = re.sub(r'[^a-z]', '', name)

    for suf in ["jr", "sr", "ii", "iii", "iv"]:
        if name.endswith(suf):
            name = name[:-len(suf)]

    return name

# =====================================================
# 3. 整理 Seasons_Stats
# =====================================================
df_season = df_season.dropna(subset=["Player", "Year"]).copy()
df_season["Year"] = pd.to_numeric(df_season["Year"], errors="coerce")
df_season = df_season.dropna(subset=["Year"]).copy()
df_season["Year"] = df_season["Year"].astype(int)

df_season["clean_name"] = df_season["Player"].apply(clean_name)

# =====================================================
# 4. 找出每位球員進聯盟第一年
# =====================================================
player_start = (
    df_season.groupby("clean_name", as_index=False)["Year"]
    .min()
    .rename(columns={"Year": "year_start"})
)

# 只抓 2011~2015 進聯盟球員
target_players = player_start[
    (player_start["year_start"] >= 2011) &
    (player_start["year_start"] <= 2015)
].copy()

target_names = set(target_players["clean_name"])

# =====================================================
# 5. 只保留目標球員資料
# =====================================================
df_target = df_season[df_season["clean_name"].isin(target_names)].copy()
df_target = df_target.merge(
    target_players,
    on="clean_name",
    how="left"
)

# =====================================================
# 6. 每季若有多筆資料：TOT 優先，否則 G 最大
# =====================================================
def choose_best_row(df_group):
    if "Tm" in df_group.columns:
        tot = df_group[df_group["Tm"] == "TOT"]
        if not tot.empty:
            return tot.iloc[0]

    if "G" in df_group.columns:
        temp = df_group.copy()
        temp["G_num"] = pd.to_numeric(temp["G"], errors="coerce")
        temp = temp.sort_values("G_num", ascending=False)
        return temp.iloc[0]

    return df_group.iloc[0]

# =====================================================
# 7. 建立重建資料（前兩年平均 + Year3_WS + Year4_WS）
# =====================================================
processed_rows = []

for clean_key, group in df_target.groupby("clean_name"):

    start_y = int(group["year_start"].iloc[0])
    player_name = group["Player"].iloc[0]

    # 每年只留一筆
    year_map = {
        int(y): choose_best_row(g)
        for y, g in group.groupby("Year")
    }

    y1 = year_map.get(start_y)
    y2 = year_map.get(start_y + 1)

    active = [x for x in [y1, y2] if x is not None]

    if len(active) == 0:
        continue

    # 前兩年平均（只數值欄位）
    avg_stats = pd.DataFrame(active).mean(numeric_only=True).to_dict()

    avg_stats["Player"] = player_name
    avg_stats["clean_name"] = clean_key
    avg_stats["year_start"] = start_y
    avg_stats["season"] = len(active)

    # Pos
    avg_stats["Pos"] = y1["Pos"] if y1 is not None and "Pos" in y1 else np.nan

    # Team
    if any((x is not None and "Tm" in x and x["Tm"] == "TOT") for x in active):
        avg_stats["Tm"] = "TOT"
    else:
        avg_stats["Tm"] = y1["Tm"] if y1 is not None and "Tm" in y1 else np.nan

    # 第三年、第四年 Win Shares
    y3 = year_map.get(start_y + 2)
    y4 = year_map.get(start_y + 3)

    avg_stats["Year3_WS"] = (
        y3["WS"] if y3 is not None and "WS" in y3 else np.nan
    )

    avg_stats["Year4_WS"] = (
        y4["WS"] if y4 is not None and "WS" in y4 else np.nan
    )

    processed_rows.append(avg_stats)

df_new = pd.DataFrame(processed_rows)

# =====================================================
# 8. Position Dummy Variables
# =====================================================
for pos in ["C", "PF", "PG", "SF", "SG"]:
    df_new[f"Pos_{pos}"] = df_new["Pos"].apply(
        lambda x: 1 if pd.notna(x) and pos in str(x) else 0
    )

# =====================================================
# 9. 對齊主檔欄位
# =====================================================
df_main["clean_name"] = df_main["Player"].apply(clean_name)

final_cols = df_main.columns.tolist()

# 主檔有的欄位，新資料沒有就補 NaN
for col in final_cols:
    if col not in df_new.columns:
        df_new[col] = np.nan

# 主檔欄位順序
df_new = df_new[final_cols].copy()

# =====================================================
# 10. 移除舊的 2011~2015 球員，再放入新資料
# =====================================================
replace_names = set(df_new["clean_name"].dropna().unique())

df_keep = df_main[
    ~df_main["clean_name"].isin(replace_names)
].copy()

df_final = pd.concat([df_keep, df_new], ignore_index=True)

# 去重（保留新資料）
df_final = df_final.drop_duplicates(
    subset=["clean_name"],
    keep="last"
).copy()

# =====================================================
# 11. 清理欄位並輸出
# =====================================================
if "clean_name" in df_final.columns:
    df_final = df_final.drop(columns=["clean_name"])

df_final.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"完成更新：已重建 {len(df_new)} 位 2011-2015 進聯盟球員")
print(f"輸出檔案：{OUTPUT_FILE}")

完成更新：已重建 375 位 2011-2015 進聯盟球員
輸出檔案：NBA_Normalized_Final.csv


In [4]:
import pandas as pd
import numpy as np
import time
import random
import re

# 1. 設定標頭偽裝
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def get_correct_table(tables):
    """從多個表格中找出真正包含球員數據的那一個"""
    for df in tables:
        if 'Player' in df.columns and ('Tm' in df.columns or 'Team' in df.columns):
            # 如果有 'Team' 改名為 'Tm' 以對齊格式
            if 'Team' in df.columns:
                df = df.rename(columns={'Team': 'Tm'})
            return df
    return None

def clean_player_name(name):
    """移除名字後面的特殊字元，例如名人堂的星號 '*'"""
    if pd.isna(name): return name
    return str(name).replace('*', '').strip()

def fetch_year_data(year):
    """抓取指定年份數據，增加錯誤處理與自動識別"""
    print(f"--- 正在處理 {year} 年數據 ---")
    
    # --- 抓取基礎數據 (Per Game) ---
    url_pg = f"https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html"
    tables_pg = pd.read_html(url_pg)
    df_pg = get_correct_table(tables_pg)
    
    if df_pg is None:
        raise ValueError(f"無法在 {year} 年基礎數據頁面找到正確表格")
    
    df_pg = df_pg[df_pg['Player'] != 'Player'].copy()
    df_pg['Player'] = df_pg['Player'].apply(clean_player_name)

    # 隨機延遲 15-30 秒
    wait = random.uniform(18, 35)
    print(f"安全等待 {wait:.2f} 秒...")
    time.sleep(wait)
    
    # --- 抓取進階數據 (Advanced) ---
    url_adv = f"https://www.basketball-reference.com/leagues/NBA_{year}_advanced.html"
    tables_adv = pd.read_html(url_adv)
    df_adv = get_correct_table(tables_adv)
    
    if df_adv is None:
        raise ValueError(f"無法在 {year} 年進階數據頁面找到正確表格")
    
    df_adv = df_adv[df_adv['Player'] != 'Player'].copy()
    df_adv['Player'] = df_adv['Player'].apply(clean_player_name)
    
    # --- 合併數據 ---
    # 找出 df_adv 中 df_pg 沒有的欄位 (例如 WS, BPM)
    cols_to_use = df_adv.columns.difference(df_pg.columns).tolist() + ['Player', 'Tm']
    df_year = pd.merge(df_pg, df_adv[cols_to_use], on=['Player', 'Tm'], how='left')
    df_year['Year'] = year
    
    return df_year

# ---------------------------------------------------------
# 執行抓取流程 (建議先抓 2017-2022)
# ---------------------------------------------------------
years_to_crawl = range(2017, 2023)
all_data_list = []

for y in years_to_crawl:
    try:
        data = fetch_year_data(y)
        all_data_list.append(data)
    except Exception as e:
        print(f"{y} 年執行時發生問題: {e}")

if all_data_list:
    df_raw = pd.concat(all_data_list, ignore_index=True)

    # 轉換所有數值欄位
    numeric_cols = df_raw.columns.drop(['Player', 'Pos', 'Tm'])
    for col in numeric_cols:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

    # --- 後續處理：前兩年平均與生涯數據 (逻辑與之前相同) ---
    player_start = df_raw.groupby('Player')['Year'].min().reset_index()
    player_start.columns = ['Player', 'year_start']

    df_target = df_raw.merge(player_start, on='Player')
    
    processed_rows = []
    for player, group in df_target.groupby('Player'):
        start_y = group['year_start'].iloc[0]
        # (這裡接續之前的 TOT 處理與平均值計算邏輯...)
        # ... [代碼省略，請延用前一段腳本的聚合部分] ...
        pass

    print("數據抓取與初步整合完成！")
else:
    print("未能抓取到任何數據，請檢查網路連線或稍後再試。")

--- 正在處理 2017 年數據 ---
安全等待 29.27 秒...
--- 正在處理 2018 年數據 ---
安全等待 20.25 秒...
--- 正在處理 2019 年數據 ---
安全等待 25.05 秒...
--- 正在處理 2020 年數據 ---
安全等待 25.69 秒...
--- 正在處理 2021 年數據 ---
安全等待 18.01 秒...
--- 正在處理 2022 年數據 ---
安全等待 27.91 秒...
數據抓取與初步整合完成！


In [6]:
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 1. 重新聚合數據：修正 season 邏輯
# ---------------------------------------------------------
processed_rows = []

# 確保數值欄位正確 (df_raw 是你爬蟲抓下來的原始大表)
numeric_cols = df_raw.columns.drop(['Player', 'Pos', 'Tm'])
for col in numeric_cols:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

# 再次確認每位球員的第一年
player_start = df_raw.groupby('Player')['Year'].min().reset_index()
player_start.columns = ['Player', 'year_start']
df_all = df_raw.merge(player_start, on='Player')

print("正在重新計算數據（修正 Season 欄位與生涯總計）...")

for player, group in df_all.groupby('Player'):
    start_y = group['year_start'].iloc[0]
    
    # 處理交易邏輯 (TOT)
    def get_tot_row(g):
        if len(g) > 1:
            tot = g[g['Tm'] == 'TOT']
            return tot.iloc[0] if not tot.empty else g.iloc[0]
        return g.iloc[0]

    year_map = {int(y): get_tot_row(y_g) for y, y_g in group.groupby('Year')}
    
    # --- A. 前兩年數據平均 (模型輸入特徵) ---
    y1 = year_map.get(int(start_y))
    y2 = year_map.get(int(start_y + 1))
    active_for_avg = [s for s in [y1, y2] if s is not None]
    
    if not active_for_avg:
        continue
    
    # 取前兩年數值平均
    row = pd.DataFrame(active_for_avg).mean(numeric_only=True).to_dict()
    
    # --- B. 修正後的 Season 欄位 (生涯總年數) ---
    # 改為計算該球員在數據庫中總共出現的年份數量
    row['season'] = group['Year'].nunique()
    
    # --- C. 基本資訊與生涯總數據 ---
    row['Player'] = player
    row['year_start'] = start_y
    row['Pos'] = active_for_avg[0]['Pos']
    
    # 生涯累計 (Advanced 欄位)
    row['Advanced_WS'] = group['WS'].sum()
    row['Advanced_VORP'] = group['VORP'].sum()
    row['Advanced_BPM'] = group['BPM'].mean()
    row['Advanced_WS/48'] = group['WS/48'].mean()
    
    # 預測目標 (第三年表現)
    y3 = year_map.get(int(start_y + 2))
    row['Year3_WS'] = y3['WS'] if y3 is not None else np.nan
    
    processed_rows.append(row)

df_modern = pd.DataFrame(processed_rows)

# ---------------------------------------------------------
# 2. 格式對齊與寫入
# ---------------------------------------------------------
# 位置 One-hot
for pos in ['C', 'PF', 'PG', 'SF', 'SG']:
    df_modern[f'Pos_{pos}'] = df_modern['Pos'].apply(lambda x: 1 if str(x).find(pos) != -1 else 0)

# 乾淨名稱
df_modern['clean_name'] = df_modern['Player'].apply(lambda x: re.sub(r'[^a-z]', '', str(x).lower()))

# 讀取原始檔案並對齊
df_old = pd.read_csv('NBA_Normalized_Final.csv')

# 補齊缺失欄位
for col in df_old.columns:
    if col not in df_modern.columns:
        df_modern[col] = np.nan

# 確保順序一致並合併
df_modern_final = df_modern[df_old.columns]
df_combined = pd.concat([df_old, df_modern_final], ignore_index=True)

# 儲存
df_combined.to_csv('NBA_Normalized_Final_Updated.csv', index=False)

print(f"修正完成！已寫入 {len(df_modern_final)} 位球員。")
print("現在 'season' 欄位已正確反映球員的生涯總年數。")

正在重新計算數據（修正 Season 欄位與生涯總計）...
修正完成！已寫入 1085 位球員。
現在 'season' 欄位已正確反映球員的生涯總年數。


In [56]:
import pandas as pd
import re

# 1. 讀取檔案
main_file = "NBA_Normalized_Final.csv"
draft_file = "sportsref_download_2017.xlsx"

df_main = pd.read_csv(main_file)
df_draft = pd.read_excel(draft_file)

# 2. 名字標準化函數
def get_clean_key(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = re.sub(r'[^a-z]', '', name)  # 移除所有非字母字元
    for suf in ["jr", "sr", "ii", "iii", "iv"]:
        if name.endswith(suf):
            name = name[:-len(suf)]
    return name

df_main["merge_key"] = df_main["Player"].apply(get_clean_key)
df_draft["merge_key"] = df_draft["Player"].apply(get_clean_key)

# 3. 檢查必要欄位
required_main_cols = ["Player"]
required_draft_cols = ["Player", "Pk"]

for col in required_main_cols:
    if col not in df_main.columns:
        raise KeyError(f"主檔缺少必要欄位: {col}")

for col in required_draft_cols:
    if col not in df_draft.columns:
        raise KeyError(f"選秀檔缺少必要欄位: {col}")

# 4. 從選秀檔提取選秀順位
# 如果你只要選秀順位，就只保留 merge_key 與 Pk
df_draft_sub = (
    df_draft[["merge_key", "Player", "Pk"]]
    .dropna(subset=["Pk"])
    .drop_duplicates(subset=["merge_key"], keep="first")
    .rename(columns={"Pk": "DraftRank_new", "Player": "DraftPlayer"})
)

# 5. 合併資料
df_merged = df_main.merge(df_draft_sub, on="merge_key", how="left")

# 6. 只在有抓到選秀順位時更新 DraftRank
mask = df_merged["DraftRank_new"].notna()

# 若主檔沒有 DraftRank 欄位，先建立
if "DraftRank" not in df_merged.columns:
    df_merged["DraftRank"] = pd.NA

df_merged.loc[mask, "DraftRank"] = df_merged.loc[mask, "DraftRank_new"]

print(f"✅ 成功提取並更新選秀順位，共 {mask.sum()} 筆。")

# 7. 輸出一些匹配結果供檢查
matched_preview = df_merged.loc[mask, ["Player", "DraftPlayer", "DraftRank"]].head(20)
print("\n--- 前 20 筆匹配結果 ---")
print(matched_preview)

# 8. 清理輔助欄位並存回主檔
drop_cols = ["merge_key", "DraftRank_new", "DraftPlayer"]
df_final = df_merged.drop(columns=[c for c in drop_cols if c in df_merged.columns])

df_final.to_csv(main_file, index=False, encoding="utf-8-sig")

# 9. 驗證範例
kyrie = df_final[df_final["Player"].str.contains("Kyrie Irving", na=False, case=False)]
print("\n--- Kyrie Irving 驗證結果 ---")
cols_to_show = [c for c in ["Player", "DraftRank"] if c in df_final.columns]
print(kyrie[cols_to_show])

✅ 成功提取並更新選秀順位，共 0 筆。

--- 前 20 筆匹配結果 ---
Empty DataFrame
Columns: [Player, DraftPlayer, DraftRank]
Index: []

--- Kyrie Irving 驗證結果 ---
           Player  DraftRank
636  Kyrie Irving        1.0


In [25]:
import pandas as pd
import numpy as np

def process_nba_finances(file_path):
    # 1. 讀取檔案
    df = pd.read_csv(file_path)
    
    # 2. 定義 2011-2022 的薪資上限資料 (單位：百萬美金)
    cap_data = {
        2011: 58.04, 2012: 58.04, 2013: 58.68, 2014: 63.07,
        2015: 70.00, 2016: 94.14, 2017: 99.09, 2018: 101.87,
        2019: 109.14, 2020: 109.14, 2021: 112.41, 2022: 123.65
    }
    
    # 3. 設定計算範圍 (2011-2022)
    mask = (df['year_start'] >= 2011) & (df['year_start'] <= 2022)
    
    print(f"正在處理 {mask.sum()} 筆近代球員數據...")
    
    # 4. 補齊 Salary_Cap_At_Start 與 BRI_Factor
    # 使用 map 函數快速對應年份與薪資上限
    df.loc[mask, 'Salary_Cap_At_Start'] = df.loc[mask, 'year_start'].map(cap_data)
    df.loc[mask, 'BRI_Factor'] = 1.25  # 近代標準值
    
    # 5. 定義 Salary_Percentage 計算邏輯 (基於 DraftRank)
    def calculate_salary_pct(rank):
        if pd.isna(rank):
            return 1.1  # 落選球員預設底薪比例
        rank = int(rank)
        if 1 <= rank <= 30:
            # 首輪球員公式：從第一順位 8.0% 線性遞減到第 30 順位 1.5%
            return 8.0 - (rank - 1) * (6.5 / 29)
        else:
            # 二輪球員統一底薪比例
            return 1.1

    # 套用公式
    df.loc[mask, 'Salary_Percentage'] = df.loc[mask, 'DraftRank'].apply(calculate_salary_pct)
    
    # 6. 計算 ROI (產出 / 成本)
    # 確保 Advanced_WS 是數值型態
    df['Advanced_WS'] = pd.to_numeric(df['Advanced_WS'], errors='coerce')
    df.loc[mask, 'ROI'] = df.loc[mask, 'Advanced_WS'] / df.loc[mask, 'Salary_Percentage']
    
    # 7. 儲存結果
    output_name = file_path.replace(".csv", "_Processed.csv")
    df.to_csv(output_name, index=False)
    
    print(f"✅ 計算完成！檔案已儲存為: {output_name}")
    return df

# 執行程式
# 請確保檔案 NBA_Normalized_Final_Updated.csv 在同一個資料夾下
processed_df = process_nba_finances('NBA_Normalized_Final_Updated.csv')

# 預覽 Kyrie Irving 的計算結果
kyrie = processed_df[processed_df['Player'].str.contains('Kyrie Irving', na=False)]
print("\n[驗證] Kyrie Irving 的計算結果：")
print(kyrie[['Player', 'year_start', 'DraftRank', 'Salary_Percentage', 'ROI']])

正在處理 1541 筆近代球員數據...
✅ 計算完成！檔案已儲存為: NBA_Normalized_Final_Updated_Processed.csv

[驗證] Kyrie Irving 的計算結果：
            Player  year_start  DraftRank  Salary_Percentage     ROI
880   Kyrie Irving      2011.0        1.0                8.0  5.0625
1441  Kyrie Irving      2011.0        1.0                8.0     NaN


In [35]:
import pandas as pd
import requests
import re
import time
import random
from bs4 import BeautifulSoup

def fetch_bbr_advanced(year):
    """
    精準抓取 BBR 進階數據，解決 403 封鎖、隱藏註解與表格偏移問題
    """
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_advanced.html"
    print(f"--- 正在嘗試抓取 {year} 年數據 ---")
    
    # 偽裝成真實瀏覽器，降低被 403 封鎖的機率
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/115.0.0.0 Safari/537.36'
    }
    
    try:
        response = requests.get(url, headers=headers)
        if response.status_code != 200:
            print(f"❌ 無法存取 {year} 年 (狀態碼: {response.status_code})。請更換網路 IP 或稍後再試。")
            return None
        
        # 關鍵：有些表格藏在註解裡，必須用正則表達式拆開
        soup = BeautifulSoup(response.text, 'html.parser')
        placeholder = soup.find('div', {'id': 'all_advanced_stats'})
        
        if placeholder:
            # 找到註解內容並重新解析
            comment = placeholder.find(string=lambda text: isinstance(text, re.Pattern) or "table" in text)
            if comment:
                table_html = str(comment).replace('', '')
                df = pd.read_html(table_html)[0]
            else:
                df = pd.read_html(response.text)[0]
        else:
            df = pd.read_html(response.text)[0]

        # 數據清洗：排除標題行、移除名字星號、處理交易(TOT)
        df = df[df['Player'] != 'Player'].copy()
        df['Player'] = df['Player'].str.replace('*', '', regex=False)
        df['WS'] = pd.to_numeric(df['WS'], errors='coerce')
        df['WS/48'] = pd.to_numeric(df['WS/48'], errors='coerce')
        
        # 處理交易：優先保留 TOT (全季總計)
        if 'Tm' in df.columns:
            df['is_tot'] = (df['Tm'] == 'TOT').astype(int)
            df = df.sort_values('is_tot', ascending=False).drop_duplicates('Player')
        
        print(f"✅ 成功抓取 {year} 年，球員數：{len(df)}")
        return df[['Player', 'WS', 'WS/48']]
        
    except Exception as e:
        print(f"⚠️ 抓取 {year} 年發生錯誤: {e}")
        return None

# ==========================================
# 執行流程
# ==========================================

# 1. 讀取你的檔案
df_main = pd.read_csv('NBA_Normalized_Final.csv')

# 2. 確定要抓的年份 (2011新秀的第3年是2013賽季，以此類推)
# 建議分批抓取，避免被封鎖
years_to_crawl = range(2013, 2025) 
data_pool = {}

for y in years_to_crawl:
    df_yr = fetch_bbr_advanced(y)
    if df_yr is not None:
        data_pool[y] = df_yr.set_index('Player').to_dict('index')
    
    # 非常重要：休息 20-30 秒。BBR 現在每分鐘限制 20 次請求。
    wait = random.uniform(20, 30)
    print(f"等待 {wait:.1f} 秒...")
    time.sleep(wait)

# 3. 回填數據
print("\n正在回填 Year3/Year4 數據...")
def fill_stats(row):
    p = row['Player']
    sy = row['year_start']
    if pd.isna(sy) or pd.isna(p): return row
    
    y3, y4 = int(sy + 2), int(sy + 3)
    
    # 補第 3 年
    if y3 in data_pool and p in data_pool[y3]:
        row['Year3_WS'] = data_pool[y3][p]['WS']
        row['Year3_WS48'] = data_pool[y3][p]['WS/48']
        
    # 補第 4 年
    if y4 in data_pool and p in data_pool[y4]:
        row['Year4_WS'] = data_pool[y4][p]['WS']
        row['Year4_WS48'] = data_pool[y4][p]['WS/48']
    return row

df_main = df_main.apply(fill_stats, axis=1)

# 4. 存檔
df_main.to_csv('NBA_Normalized_Final_Updated_Finished.csv', index=False)
print("\n🎉 補完任務完成！請檢查 NBA_Normalized_Final_Updated_Finished.csv")

--- 正在嘗試抓取 2013 年數據 ---
❌ 無法存取 2013 年 (狀態碼: 403)。請更換網路 IP 或稍後再試。
等待 22.6 秒...
--- 正在嘗試抓取 2014 年數據 ---
❌ 無法存取 2014 年 (狀態碼: 403)。請更換網路 IP 或稍後再試。
等待 24.4 秒...


KeyboardInterrupt: 

In [48]:
import pandas as pd
import numpy as np
import re

# ==========================================
# 1. 讀取檔案
# ==========================================
print("正在讀取檔案...")
df_main = pd.read_csv('NBA_Normalized_Final.csv', encoding='utf-8-sig')
df_adv = pd.read_csv('NBA_Advanced_Stats_2002-2022.csv', encoding='utf-8-sig')

def normalize_name(s):
    return re.sub(r'[^a-z]', '', str(s).lower())

def extract_name(s):
    if isinstance(s, str) and '-' in s:
        return s.split('-', 1)[1]
    return s

df_adv['c_name'] = df_adv['year-name'].apply(extract_name).apply(normalize_name)
df_main['c_name'] = df_main['Player'].apply(normalize_name)

# 處理交易 (優先取 TOT)
df_adv['is_tot'] = (df_adv['Tm'] == 'TOT').astype(int)
df_adv_clean = df_adv.sort_values(['c_name', 'year', 'is_tot'], ascending=[True, True, False])
df_adv_clean = df_adv_clean.drop_duplicates(subset=['c_name', 'year'], keep='first')

# 定義要平均的所有進階數據欄位
stats_to_avg = [
    'PER', 'TS%', '3PAr', 'FTr', 'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 
    'TOV%', 'USG%', 'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP'
]

# ==========================================
# 2. 提取並計算 2011-2021 球員數據
# ==========================================
processed_list = []
processed_names = set()

print("正在計算球員的前兩年平均與三、四年目標值...")

for name_key, group in df_adv_clean.groupby('c_name'):
    group = group.sort_values('year')
    start_yr = int(group['year'].min())
    
    # 只處理 2011-2021 梯次
    if 2011 <= start_yr <= 2021:
        # 前兩年 (Year 1 & 2)
        y12 = group[group['year'].isin([start_yr, start_yr + 1])]
        # 第三、四年 (Year 3 & 4)
        y3 = group[group['year'] == start_yr + 2]
        y4 = group[group['year'] == start_yr + 3]
        
        new_row = {
            'c_name': name_key,
            'Player': group.iloc[0]['year-name'].split('-',1)[1],
            'year_start': start_yr,
            'Pos': group.iloc[0]['Pos']
        }
        
        # A. 填入前兩年平均數據 (寫入主檔的所有對應欄位)
        if not y12.empty:
            avg_vals = y12[stats_to_avg].mean()
            for col in stats_to_avg:
                new_row[col] = avg_vals[col]
                # 同時映射到 "Advanced_" 開頭的欄位
                adv_col_name = f"Advanced_{col}"
                new_row[adv_col_name] = avg_vals[col]

        # B. 填入第三、四年目標值
        new_row['Year3_WS'] = y3['WS'].values[0] if not y3.empty else np.nan
        new_row['Year3_WS48'] = y3['WS/48'].values[0] if not y3.empty else np.nan
        new_row['Year4_WS'] = y4['WS'].values[0] if not y4.empty else np.nan
        new_row['Year4_WS48'] = y4['WS/48'].values[0] if not y4.empty else np.nan
        
        processed_list.append(new_row)
        processed_names.add(name_key)

df_processed = pd.DataFrame(processed_list)

# ==========================================
# 3. 刪除舊資料並重寫
# ==========================================
print(f"正在從主檔移除並更新 {len(processed_names)} 位球員...")

# 保留主檔中原有但不屬於這批更新名單的人
df_main_preserved = df_main[~df_main['c_name'].isin(processed_names)].copy()

# 獲取主檔中這批球員的「元數據」(例如 DraftRank, Salary_Percentage)
# 這樣在重寫時才不會丟失選秀順位等資訊
meta_cols = ['c_name', 'DraftRank', 'Is_Undrafted', 'Salary_Percentage', 'ROI', 'Salary_Cap_At_Start', 'BRI_Factor']
df_meta = df_main[df_main['c_name'].isin(processed_names)][meta_cols].drop_duplicates('c_name')

# 合併數據與元數據
df_processed_final = pd.merge(df_processed, df_meta, on='c_name', how='left')

# 最終合併所有球員
df_final = pd.concat([df_main_preserved, df_processed_final], ignore_index=True)

# 移除輔助欄位並存檔
df_final.drop(columns=['c_name'], inplace=True, errors='ignore')
df_final.to_csv('NBA_Normalized_Final.csv', index=False, encoding='utf-8-sig')

print("\n🎉 處理完成！")
print(f"1. 2011-2021 梯次球員已完全根據進階數據表更新。")
print(f"2. 前兩年平均已補齊，包含 WS 與進階指標。")
print(f"3. 第三、四年 WS 數據已根據 2022 檔案限制盡可能抓取。")
print(f"檔案已儲存為: NBA_Final_Rewritten_All_Stats.csv")

正在讀取檔案...
正在計算球員的前兩年平均與三、四年目標值...
正在從主檔移除並更新 980 位球員...

🎉 處理完成！
1. 2011-2021 梯次球員已完全根據進階數據表更新。
2. 前兩年平均已補齊，包含 WS 與進階指標。
3. 第三、四年 WS 數據已根據 2022 檔案限制盡可能抓取。
檔案已儲存為: NBA_Final_Rewritten_All_Stats.csv


In [50]:
import os
import re
import time
import random
from io import StringIO

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

# ==========================================
# 1. 基本設定
# ==========================================
INPUT_FILE = "NBA_Normalized_Final.csv"
OUTPUT_FILE = "NBA_Normalized_Final.csv"
CACHE_DIR = "br_cache"

os.makedirs(CACHE_DIR, exist_ok=True)

shooting_cols = [
    'FG', 'FGA', 'FG%', '3P', '3PA', '3P%', '2P', '2PA', '2P%',
    'eFG%', 'FT', 'FTA', 'FT%', '3PAr'
]

YEARS = range(2011, 2026)

def normalize_name(s):
    return re.sub(r'[^a-z]', '', str(s).lower())

# ==========================================
# 2. 建立 Session：重試、退避、連線重用
# ==========================================
def build_session():
    session = requests.Session()

    retry = Retry(
        total=5,
        read=5,
        connect=5,
        backoff_factor=2,  # 1, 2, 4, 8...
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )

    adapter = HTTPAdapter(max_retries=retry, pool_connections=10, pool_maxsize=10)
    session.mount("https://", adapter)
    session.mount("http://", adapter)

    session.headers.update({
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/122.0.0.0 Safari/537.36"
        ),
        "Accept-Language": "en-US,en;q=0.9",
        "Referer": "https://www.basketball-reference.com/",
        "Connection": "keep-alive",
    })
    return session

# ==========================================
# 3. 抓取單一年資料：先讀快取，沒有才打網站
# ==========================================
def fetch_nba_stats(year, session, force_refresh=False):
    cache_path = os.path.join(CACHE_DIR, f"NBA_{year}_per_game.html")
    url = f"https://www.basketball-reference.com/leagues/NBA_{year}_per_game.html"

    # 先讀本地快取，避免重複打網站
    if os.path.exists(cache_path) and not force_refresh:
        with open(cache_path, "r", encoding="utf-8") as f:
            html = f.read()
    else:
        print(f"抓取 {year} 年資料: {url}")

        # 加入較保守的隨機等待，避免固定節奏
        time.sleep(random.uniform(4, 8))

        resp = session.get(url, timeout=30)

        # 如果真的 429，再做額外等待
        if resp.status_code == 429:
            wait_time = random.uniform(60, 120)
            print(f"遇到 429，休息 {wait_time:.1f} 秒後重試 {year}")
            time.sleep(wait_time)

            resp = session.get(url, timeout=30)

        resp.raise_for_status()
        html = resp.text

        # 存快取
        with open(cache_path, "w", encoding="utf-8") as f:
            f.write(html)

    # 讀表
    tables = pd.read_html(StringIO(html))
    df = tables[0]

    # 清掉重複表頭列
    df = df[df["Player"] != "Player"].copy()

    # 名字標準化
    df["c_name"] = df["Player"].apply(normalize_name)

    # 處理交易球員：TOT 優先
    df["is_tot"] = (df["Tm"] == "TOT").astype(int)
    df = (
        df.sort_values(["c_name", "is_tot"], ascending=[True, False])
          .drop_duplicates("c_name", keep="first")
          .copy()
    )

    # 只保留需要欄位
    keep_cols = ["Player", "Tm", "c_name"] + [c for c in shooting_cols if c in df.columns]
    df = df[keep_cols].copy()

    # 轉數值
    for col in shooting_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

# ==========================================
# 4. 一次抓完所有年份，做成總表
# ==========================================
def build_reference_table(years, force_refresh=False):
    session = build_session()
    frames = []

    for year in years:
        try:
            df_year = fetch_nba_stats(year, session=session, force_refresh=force_refresh)
            df_year["season_year"] = year
            frames.append(df_year)
            print(f"完成 {year}")
        except Exception as e:
            print(f"{year} 失敗: {e}")

    if not frames:
        raise RuntimeError("所有年度資料都抓失敗，無法繼續")

    ref_all = pd.concat(frames, ignore_index=True)
    return ref_all

# ==========================================
# 5. 只計算每位球員前兩年的平均
# ==========================================
def prepare_two_year_average(ref_all, df_main):
    players = df_main[["Player", "year_start"]].copy()
    players["c_name"] = players["Player"].apply(normalize_name)
    players = players.dropna(subset=["year_start"]).copy()
    players["year_start"] = players["year_start"].astype(int)

    # 只處理 2011~2021 梯次
    players = players[(players["year_start"] >= 2011) & (players["year_start"] <= 2021)].copy()

    # 展開成 start 與 start+1 兩年
    p1 = players[["c_name", "year_start"]].rename(columns={"year_start": "season_year"})
    p2 = players[["c_name", "year_start"]].copy()
    p2["season_year"] = p2["year_start"] + 1
    p2 = p2[["c_name", "season_year"]]

    target_seasons = pd.concat([p1, p2], ignore_index=True)

    # 對應 reference
    merged = target_seasons.merge(
        ref_all[["c_name", "season_year"] + shooting_cols],
        on=["c_name", "season_year"],
        how="left"
    )

    # 每個球員取前兩年平均（若只抓到一年，就保留那一年）
    avg_stats = (
        merged.groupby("c_name", as_index=False)[shooting_cols]
              .mean(numeric_only=True)
    )

    return avg_stats

# ==========================================
# 6. 主流程：merge 回主檔
# ==========================================
def main(force_refresh=False):
    df_main = pd.read_csv(INPUT_FILE)
    df_main["c_name"] = df_main["Player"].apply(normalize_name)

    ref_all = build_reference_table(YEARS, force_refresh=force_refresh)
    avg_stats = prepare_two_year_average(ref_all, df_main)

    # 刪除舊欄位再重寫
    for col in shooting_cols:
        if col in df_main.columns:
            df_main.drop(columns=[col], inplace=True)

    df_main = df_main.merge(avg_stats, on="c_name", how="left")

    df_main.drop(columns=["c_name"], inplace=True, errors="ignore")
    df_main.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print("完成，已更新前兩年平均投籃基礎數據")

if __name__ == "__main__":
    main(force_refresh=False)

抓取 2011 年資料: https://www.basketball-reference.com/leagues/NBA_2011_per_game.html
2011 失敗: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/leagues/NBA_2011_per_game.html
抓取 2012 年資料: https://www.basketball-reference.com/leagues/NBA_2012_per_game.html
2012 失敗: 403 Client Error: Forbidden for url: https://www.basketball-reference.com/leagues/NBA_2012_per_game.html
抓取 2013 年資料: https://www.basketball-reference.com/leagues/NBA_2013_per_game.html


KeyboardInterrupt: 

In [53]:
pip install nba_api pandas numpy requests

In [54]:
import os
import re
import time
import random
import numpy as np
import pandas as pd

from nba_api.stats.endpoints import leaguedashplayerstats

INPUT_FILE = "NBA_Normalized_Final.csv"
OUTPUT_FILE = "NBA_Normalized_Final.csv"
CACHE_DIR = "nba_api_cache"

os.makedirs(CACHE_DIR, exist_ok=True)

shooting_cols = [
    'FG', 'FGA', 'FG%', '3P', '3PA', '3P%',
    '2P', '2PA', '2P%', 'eFG%', 'FT', 'FTA', 'FT%', '3PAr'
]

def normalize_name(s):
    return re.sub(r'[^a-z]', '', str(s).lower())

def safe_div(a, b):
    return np.where((b == 0) | pd.isna(b), np.nan, a / b)

def fetch_nba_official_stats(season_start_year, force_refresh=False):
    """
    抓 NBA 官方 stats.nba.com 的單一球季 Per Game 傳統數據
    season_start_year=2010 -> season='2010-11'
    """
    season = f"{season_start_year}-{str(season_start_year + 1)[-2:]}"
    cache_path = os.path.join(CACHE_DIR, f"{season}_players_traditional.csv")

    if os.path.exists(cache_path) and not force_refresh:
        df = pd.read_csv(cache_path)
        return df

    print(f"抓取官方 NBA 數據: {season}")

    # 稍微放慢，避免過密
    time.sleep(random.uniform(1.5, 4.0))

    # LeagueDashPlayerStats 對應 stats.nba.com 的球員傳統數據
    result = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star="Regular Season",
        per_mode_detailed="PerGame"
    )

    df = result.get_data_frames()[0].copy()

    # 常見欄位名稱：
    # PLAYER_NAME, TEAM_ABBREVIATION, GP, MIN, FGM, FGA, FG_PCT, FG3M, FG3A, FG3_PCT, FTM, FTA, FT_PCT ...
    rename_map = {
        "PLAYER_NAME": "Player",
        "TEAM_ABBREVIATION": "Tm",
        "FGM": "FG",
        "FGA": "FGA",
        "FG_PCT": "FG%",
        "FG3M": "3P",
        "FG3A": "3PA",
        "FG3_PCT": "3P%",
        "FTM": "FT",
        "FTA": "FTA",
        "FT_PCT": "FT%",
    }

    df = df.rename(columns=rename_map)

    keep = ["Player", "Tm", "FG", "FGA", "FG%", "3P", "3PA", "3P%", "FT", "FTA", "FT%"]
    df = df[[c for c in keep if c in df.columns]].copy()

    df["c_name"] = df["Player"].apply(normalize_name)

    # 推導欄位
    df["2P"] = df["FG"] - df["3P"]
    df["2PA"] = df["FGA"] - df["3PA"]
    df["2P%"] = safe_div(df["2P"], df["2PA"])
    df["eFG%"] = safe_div(df["FG"] + 0.5 * df["3P"], df["FGA"])
    df["3PAr"] = safe_div(df["3PA"], df["FGA"])

    # 只保留需要的欄位
    df = df[["Player", "Tm", "c_name"] + shooting_cols].copy()

    df.to_csv(cache_path, index=False, encoding="utf-8-sig")
    return df

def build_reference_table(start_year=2010, end_year=2025, force_refresh=False):
    """
    start_year=2010 代表抓 2010-11
    end_year=2025 代表抓 2025-26
    """
    frames = []

    for y in range(start_year, end_year + 1):
        try:
            df_year = fetch_nba_official_stats(y, force_refresh=force_refresh)
            df_year["season_start_year"] = y
            frames.append(df_year)
            print(f"完成 {y}-{str(y+1)[-2:]}")
        except Exception as e:
            print(f"失敗 {y}-{str(y+1)[-2:]}: {e}")

    if not frames:
        raise RuntimeError("沒有成功抓到任何球季資料")

    return pd.concat(frames, ignore_index=True)

def prepare_two_year_average(df_main, ref_all):
    work = df_main[["Player", "year_start"]].copy()
    work["c_name"] = work["Player"].apply(normalize_name)
    work = work.dropna(subset=["year_start"]).copy()
    work["year_start"] = work["year_start"].astype(int)

    # 只處理 2011~2021 梯次
    work = work[(work["year_start"] >= 2011) & (work["year_start"] <= 2021)].copy()

    # 你的 year_start 看起來是選秀梯次 / 起始年
    # 若你要「前兩年 NBA 球季平均」，通常對應 year_start 當季與下一季
    s1 = work[["c_name", "year_start"]].rename(columns={"year_start": "season_start_year"})
    s2 = work[["c_name", "year_start"]].copy()
    s2["season_start_year"] = s2["year_start"] + 1
    s2 = s2[["c_name", "season_start_year"]]

    wanted = pd.concat([s1, s2], ignore_index=True)

    merged = wanted.merge(
        ref_all[["c_name", "season_start_year"] + shooting_cols],
        on=["c_name", "season_start_year"],
        how="left"
    )

    avg_stats = (
        merged.groupby("c_name", as_index=False)[shooting_cols]
        .mean(numeric_only=True)
    )

    return avg_stats

def main(force_refresh=False):
    df_main = pd.read_csv(INPUT_FILE)
    df_main["c_name"] = df_main["Player"].apply(normalize_name)

    # 若你的主資料只需要補 2011~2025 對應球季，
    # 實際上抓 2011-12 到 2022-23 可能就夠；
    # 這裡保守抓 2010-11 到 2025-26
    ref_all = build_reference_table(start_year=2010, end_year=2025, force_refresh=force_refresh)

    avg_stats = prepare_two_year_average(df_main, ref_all)

    # 刪除舊欄位後重寫
    for col in shooting_cols:
        if col in df_main.columns:
            df_main.drop(columns=[col], inplace=True)

    df_main = df_main.merge(avg_stats, on="c_name", how="left")

    df_main.drop(columns=["c_name"], inplace=True, errors="ignore")
    df_main.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

    print("完成：已用 NBA 官方資料補上投籃欄位")

if __name__ == "__main__":
    main(force_refresh=False)

抓取官方 NBA 數據: 2010-11
失敗 2010-11: ('Connection aborted.', ConnectionResetError(10054, '遠端主機已強制關閉一個現存的連線。', None, 10054, None))
抓取官方 NBA 數據: 2011-12
失敗 2011-12: ('Connection aborted.', ConnectionResetError(10054, '遠端主機已強制關閉一個現存的連線。', None, 10054, None))
抓取官方 NBA 數據: 2012-13
失敗 2012-13: ('Connection aborted.', ConnectionResetError(10054, '遠端主機已強制關閉一個現存的連線。', None, 10054, None))
抓取官方 NBA 數據: 2013-14
失敗 2013-14: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
抓取官方 NBA 數據: 2014-15
失敗 2014-15: HTTPSConnectionPool(host='stats.nba.com', port=443): Read timed out. (read timeout=30)
抓取官方 NBA 數據: 2015-16
失敗 2015-16: ('Connection aborted.', ConnectionResetError(10054, '遠端主機已強制關閉一個現存的連線。', None, 10054, None))
抓取官方 NBA 數據: 2016-17
失敗 2016-17: ('Connection aborted.', ConnectionResetError(10054, '遠端主機已強制關閉一個現存的連線。', None, 10054, None))
抓取官方 NBA 數據: 2017-18
失敗 2017-18: ('Connection aborted.', ConnectionResetError(10054, '遠端主機已強制關閉一個現存的連線。', None, 10054, None))
抓取官方

RuntimeError: 沒有成功抓到任何球季資料

In [1]:
import os
import time
import random
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CACHE_DIR = "bref_cache"
os.makedirs(CACHE_DIR, exist_ok=True)

SEASON_YEARS = list(range(2018, 2027))   # 2018 ~ 2026 season pages
DRAFT_YEARS = list(range(2011, 2023))    # 2011 ~ 2022 draft pages

def build_session():
    s = requests.Session()
    retry = Retry(
        total=2,
        connect=2,
        read=2,
        backoff_factor=2,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    s.mount("https://", adapter)
    s.mount("http://", adapter)
    s.headers.update({
        "User-Agent": "Mozilla/5.0",
        "Accept-Language": "en-US,en;q=0.9",
    })
    return s

def polite_get(session, url, cache_path, min_sleep=6, max_sleep=12):
    if os.path.exists(cache_path):
        print(f"使用快取: {cache_path}")
        return True

    # 保守速率：遠低於每分鐘 20 次
    wait = random.uniform(min_sleep, max_sleep)
    print(f"等待 {wait:.1f} 秒後抓取: {url}")
    time.sleep(wait)

    resp = session.get(url, timeout=30)

    if resp.status_code == 200:
        with open(cache_path, "w", encoding="utf-8") as f:
            f.write(resp.text)
        print(f"已儲存: {cache_path}")
        return True

    if resp.status_code in (403, 429):
        retry_after = resp.headers.get("Retry-After")
        print(f"遇到 {resp.status_code}：{url}")
        if retry_after:
            print(f"Retry-After: {retry_after}")
        print("停止抓取，避免持續撞站。")
        return False

    print(f"失敗 {resp.status_code}: {url}")
    return False

def download_missing_pages():
    session = build_session()

    # season per_game + advanced
    for y in SEASON_YEARS:
        season_pages = {
            f"NBA_{y}_per_game.html": f"https://www.basketball-reference.com/leagues/NBA_{y}_per_game.html",
            f"NBA_{y}_advanced.html": f"https://www.basketball-reference.com/leagues/NBA_{y}_advanced.html",
        }

        for filename, url in season_pages.items():
            cache_path = os.path.join(CACHE_DIR, filename)
            ok = polite_get(session, url, cache_path)
            if not ok:
                return False

    # draft pages
    for y in DRAFT_YEARS:
        filename = f"NBA_{y}_draft.html"
        url = f"https://www.basketball-reference.com/draft/NBA_{y}.html"
        cache_path = os.path.join(CACHE_DIR, filename)
        ok = polite_get(session, url, cache_path)
        if not ok:
            return False

    return True

if __name__ == "__main__":
    success = download_missing_pages()
    print("下載完成" if success else "下載中止")

等待 11.8 秒後抓取: https://www.basketball-reference.com/leagues/NBA_2018_per_game.html
遇到 403：https://www.basketball-reference.com/leagues/NBA_2018_per_game.html
停止抓取，避免持續撞站。
下載中止


In [3]:
import pandas as pd
import numpy as np
import re

# =====================================================
# 1. 檔案設定
# =====================================================
SEASON_FILE = "Seasons_Stats.csv"
MAIN_FILE = "NBA_Normalized_Final.csv"
OUTPUT_FILE = "NBA_Normalized_Final_2011_2015_updated.csv"

# =====================================================
# 2. 讀檔
# =====================================================
df_season = pd.read_csv(SEASON_FILE)
df_main = pd.read_csv(MAIN_FILE)

# =====================================================
# 3. 名字清理
# =====================================================
def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = re.sub(r"[^a-z]", "", name)
    for suf in ["jr", "sr", "ii", "iii", "iv"]:
        if name.endswith(suf):
            name = name[:-len(suf)]
    return name

df_season["clean_name"] = df_season["Player"].apply(clean_name)
df_main["clean_name"] = df_main["Player"].apply(clean_name)

# =====================================================
# 4. Year 轉數值
# =====================================================
df_season = df_season.dropna(subset=["Player", "Year"]).copy()
df_season["Year"] = pd.to_numeric(df_season["Year"], errors="coerce")
df_season = df_season.dropna(subset=["Year"]).copy()
df_season["Year"] = df_season["Year"].astype(int)

# =====================================================
# 5. 找出每位球員首次進聯盟年份
# =====================================================
player_start = (
    df_season.groupby("clean_name", as_index=False)["Year"]
    .min()
    .rename(columns={"Year": "year_start"})
)

target_players = player_start[
    (player_start["year_start"] >= 2011) &
    (player_start["year_start"] <= 2015)
].copy()

target_names = set(target_players["clean_name"])

df_target = df_season[df_season["clean_name"].isin(target_names)].copy()
df_target = df_target.merge(target_players, on="clean_name", how="left")

# =====================================================
# 6. 每季多筆資料時：TOT 優先，否則 G 最大
# =====================================================
def choose_best_row(df_group):
    if "Tm" in df_group.columns:
        tot = df_group[df_group["Tm"] == "TOT"]
        if not tot.empty:
            return tot.iloc[0]

    if "G" in df_group.columns:
        temp = df_group.copy()
        temp["G_num"] = pd.to_numeric(temp["G"], errors="coerce")
        temp = temp.sort_values("G_num", ascending=False)
        return temp.iloc[0]

    return df_group.iloc[0]

# =====================================================
# 7. 前兩年平均欄位
# =====================================================
avg_cols = [
    'G','2P%','2PA','3P','3P%','3PA','3PAr','AST','AST%','BLK','BLK%',
    'BPM','DBPM','DRB','DRB%','DWS','FG','FG%','FGA','FT','FT%','FTA',
    'FTr','OBPM','ORB','ORB%','OWS','PER','PF','PTS','STL','STL%','TOV',
    'TOV%','TRB','TRB%','TS%','USG%','VORP','WS','WS/48','eFG%'
]

if "2P" in df_target.columns:
    avg_cols = ["2P"] + avg_cols

for col in avg_cols + ["MP", "WS", "WS/48", "VORP"]:
    if col in df_target.columns:
        df_target[col] = pd.to_numeric(df_target[col], errors="coerce")

# =====================================================
# 8. 重建 2011-2015 球員資料
# =====================================================
processed_rows = []

for clean_key, group in df_target.groupby("clean_name"):
    start_y = int(group["year_start"].iloc[0])
    player_name = group["Player"].iloc[0]

    year_map = {
        int(y): choose_best_row(g)
        for y, g in group.groupby("Year")
    }

    y1 = year_map.get(start_y)
    y2 = year_map.get(start_y + 1)

    active = [x for x in [y1, y2] if x is not None]
    if not active:
        continue

    temp = pd.DataFrame(active).copy()
    usable_avg_cols = [c for c in avg_cols if c in temp.columns]
    avg_stats = temp[usable_avg_cols].mean(numeric_only=True).to_dict()

    row = {}
    row["Player"] = player_name
    row["clean_name"] = clean_key
    row["year_start"] = start_y
    row["season"] = len(group["Year"].dropna().unique())

    row["Pos"] = y1["Pos"] if y1 is not None and "Pos" in y1 else np.nan

    if any((x is not None and "Tm" in x and x["Tm"] == "TOT") for x in active):
        row["Tm"] = "TOT"
    else:
        row["Tm"] = y1["Tm"] if y1 is not None and "Tm" in y1 else np.nan

    # 前兩年平均
    for c, v in avg_stats.items():
        row[c] = v

    # 第三、四年 WS
    y3 = year_map.get(start_y + 2)
    y4 = year_map.get(start_y + 3)

    row["Year3_WS"] = pd.to_numeric(y3["WS"], errors="coerce") if y3 is not None and "WS" in y3 else np.nan
    row["Year4_WS"] = pd.to_numeric(y4["WS"], errors="coerce") if y4 is not None and "WS" in y4 else np.nan

    # 如果你主檔也有 WS/48 版本，一起補
    row["Year3_WS48"] = pd.to_numeric(y3["WS/48"], errors="coerce") if y3 is not None and "WS/48" in y3 else np.nan
    row["Year4_WS48"] = pd.to_numeric(y4["WS/48"], errors="coerce") if y4 is not None and "WS/48" in y4 else np.nan
    row["Year3_WS/48"] = row["Year3_WS48"]
    row["Year4_WS/48"] = row["Year4_WS48"]

    # 生涯 advanced
    career = group.copy()

    row["Advanced_WS"] = career["WS"].sum(skipna=True) if "WS" in career.columns else np.nan
    row["Advanced_VORP"] = career["VORP"].sum(skipna=True) if "VORP" in career.columns else np.nan

    if "WS/48" in career.columns and "MP" in career.columns:
        valid = career.dropna(subset=["WS/48", "MP"]).copy()
        valid = valid[valid["MP"] > 0]
        if len(valid) > 0:
            row["Advanced_WS/48"] = np.average(valid["WS/48"], weights=valid["MP"])
        else:
            row["Advanced_WS/48"] = np.nan
    else:
        row["Advanced_WS/48"] = np.nan

    processed_rows.append(row)

df_new = pd.DataFrame(processed_rows)

# =====================================================
# 9. Position dummy
# =====================================================
for pos in ["C", "PF", "PG", "SF", "SG"]:
    df_new[f"Pos_{pos}"] = df_new["Pos"].apply(
        lambda x: 1 if pd.notna(x) and pos in str(x) else 0
    )

# =====================================================
# 10. 對齊主檔欄位
# =====================================================
final_cols = df_main.columns.tolist()

for col in final_cols:
    if col not in df_new.columns:
        df_new[col] = np.nan

df_new = df_new[final_cols].copy()

# =====================================================
# 11. 刪除舊的 2011-2015 球員，換成新資料
# =====================================================
replace_names = set(df_new["clean_name"].dropna().unique())

df_keep = df_main[~df_main["clean_name"].isin(replace_names)].copy()
df_final = pd.concat([df_keep, df_new], ignore_index=True)
df_final = df_final.drop_duplicates(subset=["clean_name"], keep="last").copy()

# =====================================================
# 12. 輸出
# =====================================================
if "clean_name" in df_final.columns:
    # 如果你主檔本來就要保留 clean_name，可把這段刪掉
    pass

df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"完成：已更新 {len(df_new)} 位 2011-2015 進聯盟球員")
print(f"輸出檔案：{OUTPUT_FILE}")

完成：已更新 375 位 2011-2015 進聯盟球員
輸出檔案：NBA_Normalized_Final_2011_2015_updated.csv


In [8]:
import pandas as pd
import re

# 讀檔
df_main = pd.read_csv("NBA_Normalized_Final.csv")
df_avg3 = pd.read_csv("NBA_Normalized_Final_3.csv")

# 名字標準化
def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = re.sub(r"[^a-z]", "", name)
    for suf in ["jr", "sr", "ii", "iii", "iv"]:
        if name.endswith(suf):
            name = name[:-len(suf)]
    return name

df_main["merge_key"] = df_main["Player"].apply(clean_name)
df_avg3["merge_key"] = df_avg3["Player"].apply(clean_name)

# 避免新檔同一球員重複
sort_cols = [c for c in ["year_start", "Player"] if c in df_avg3.columns]
if sort_cols:
    df_avg3 = df_avg3.sort_values(sort_cols, kind="mergesort")
df_avg3 = df_avg3.drop_duplicates(subset="merge_key", keep="first").copy()

# 這些欄位改用「前三年平均」覆蓋
avg_cols = [
    'Age','G','GS','MP','2P','2P%','2PA','3P','3P%','3PA','3PAr','AST','AST%',
    'BLK','BLK%','BPM','DBPM','DRB','DRB%','DWS','FG','FG%','FGA','FT','FT%','FTA',
    'FTr','OBPM','ORB','ORB%','OWS','PER','PF','PTS','STL','STL%','TOV','TOV%','TRB',
    'TRB%','TS%','USG%','VORP','WS','WS/48','eFG%'
]

avg_cols = [c for c in avg_cols if c in df_main.columns and c in df_avg3.columns]

# 合併
df_merged = df_main.merge(
    df_avg3[["merge_key"] + avg_cols],
    on="merge_key",
    how="left",
    suffixes=("", "_new")
)

# 只覆蓋平均欄位，其他欄位保留原本
for col in avg_cols:
    new_col = f"{col}_new"
    if new_col in df_merged.columns:
        df_merged[col] = df_merged[new_col].combine_first(df_merged[col])
        df_merged.drop(columns=[new_col], inplace=True)

# 清理輔助欄位
df_merged.drop(columns=["merge_key"], inplace=True)

# 輸出
output_file = "NBA_Normalized_Final_merged_3year_avg.csv"
df_merged.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"✅ 已輸出：{output_file}")
print(f"🧾 筆數：{df_merged.shape[0]}，欄位數：{df_merged.shape[1]}")

✅ 已輸出：NBA_Normalized_Final_merged_3year_avg.csv
🧾 筆數：2648，欄位數：71


In [14]:
import pandas as pd
import re

# =====================================================
# 1. 讀檔
# =====================================================
main_file = "NBA_Normalized_Final.csv"
avg3_file = "NBA_Normalized_Final_3.csv"
output_file = "NBA_Normalized_Final_3_integrated.csv"

df_main = pd.read_csv(main_file)
df_avg3 = pd.read_csv(avg3_file)

# =====================================================
# 2. 名字標準化（用來對照）
# =====================================================
def clean_name(name):
    if pd.isna(name):
        return ""
    name = str(name).lower().strip()
    name = re.sub(r"[^a-z]", "", name)
    for suf in ["jr", "sr", "ii", "iii", "iv"]:
        if name.endswith(suf):
            name = name[:-len(suf)]
    return name

df_main["merge_key"] = df_main["Player"].apply(clean_name)
df_avg3["merge_key"] = df_avg3["Player"].apply(clean_name)

# =====================================================
# 3. 去重
# =====================================================
df_main = df_main.drop_duplicates(subset="merge_key", keep="first").copy()
df_avg3 = df_avg3.drop_duplicates(subset="merge_key", keep="first").copy()

# =====================================================
# 4. 定義平均欄位
#    這些欄位保留 avg3 的值
# =====================================================
avg_cols = [
    'Age','G','GS','MP',
    'PER','TS%','3PAr','FTr','ORB%','DRB%','TRB%','AST%','STL%','BLK%','TOV%','USG%',
    'OWS','DWS','WS','WS/48','OBPM','DBPM','BPM','VORP',
    'FG','FGA','FG%','3P','3PA','3P%','2P','2PA','2P%','eFG%','FT','FTA','FT%',
    'ORB','DRB','TRB','AST','STL','BLK','TOV','PF','PTS'
]

avg_cols = [c for c in avg_cols if c in df_avg3.columns]

# =====================================================
# 5. 非平均欄位：從主檔貼到 avg3
#    year_3 / year_4 會自動包含在這裡
# =====================================================
non_avg_cols = [
    c for c in df_main.columns
    if c not in avg_cols and c != "merge_key"
]

# 為了避免同名衝突，先改名
rename_map = {c: f"{c}_src" for c in non_avg_cols}
df_main_non_avg = df_main[["merge_key"] + non_avg_cols].rename(columns=rename_map).copy()

# =====================================================
# 6. 合併
# =====================================================
df_result = df_avg3.merge(
    df_main_non_avg,
    on="merge_key",
    how="left"
)

# =====================================================
# 7. 強制把主檔的非平均欄位貼過去
#    若 avg3 原本有同名欄位，也直接用主檔覆蓋
# =====================================================
for col in non_avg_cols:
    src_col = f"{col}_src"
    if src_col in df_result.columns:
        df_result[col] = df_result[src_col]
        df_result.drop(columns=[src_col], inplace=True)

# =====================================================
# 8. 欄位順序：盡量對齊主檔
# =====================================================
final_cols = [c for c in df_main.columns if c != "merge_key"]
extra_cols = [c for c in df_result.columns if c not in final_cols and c != "merge_key"]

ordered_cols = [c for c in final_cols if c in df_result.columns] + extra_cols
df_result = df_result[ordered_cols].copy()

# =====================================================
# 9. 輸出
# =====================================================
df_result.to_csv(output_file, index=False, encoding="utf-8-sig")

print(f"✅ 已輸出：{output_file}")
print(f"🧾 筆數：{df_result.shape[0]}，欄位數：{df_result.shape[1]}")

# 額外確認 year_3 / year_4 是否存在
check_cols = [c for c in ["year_3", "year_4"] if c in df_result.columns]
print("已包含欄位：", check_cols)

✅ 已輸出：NBA_Normalized_Final_3_integrated.csv
🧾 筆數：2046，欄位數：73
已包含欄位： ['year_3', 'year_4']


In [15]:
import pandas as pd

# 1. 讀取主檔案
df_main = pd.read_csv("NBA_Normalized_Final.csv")

# 2. 數據清洗：確保 year_4 格式統一 (轉小寫、去空格)
df_main['year_4_clean'] = df_main['year_4'].astype(str).str.strip().str.lower()

# 3. 篩選出所有 year_4 是 'no' 的球員
# 注意：這裡會自動排除那些 year_4 是 'yes' 或 NaN (資料缺失) 的人
declined_players = df_main[df_main['year_4_clean'] == 'no'].copy()

# 4. 計算這些人 Year3_WS 的平均值
# 我們先將 Year3_WS 轉為數值，避免髒資料報錯
declined_players['Year3_WS'] = pd.to_numeric(declined_players['Year3_WS'], errors='coerce')

# 排除 Year3_WS 剛好也是空的球員 (因為沒打當然沒數據)
valid_data = declined_players.dropna(subset=['Year3_WS'])

avg_ws_declined = valid_data['Year3_WS'].mean()
count_declined = len(valid_data)

# 5. 輸出結果
print(f"=== 決策分析報告 ===")
print(f"實際球隊『未執行』第四年期權的球員總數 (有數據者): {count_declined} 人")
print(f"這些球員在第三年的平均 Win Shares (WS): {avg_ws_declined:.4f}")

# 進一步看看這些人的分布，幫助你設定 ws_floor
print("\n--- 這些球員的 WS 統計分布 ---")
print(valid_data['Year3_WS'].describe())

# 列出前幾名作為參考
print("\n--- 範例名單 (前 5 名) ---")
print(valid_data[['Player', 'year_start', 'Year3_WS']].head())

=== 決策分析報告 ===
實際球隊『未執行』第四年期權的球員總數 (有數據者): 56 人
這些球員在第三年的平均 Win Shares (WS): 1.2250

--- 這些球員的 WS 統計分布 ---
count    56.000000
mean      1.225000
std       2.543387
min      -1.300000
25%       0.000000
50%       0.400000
75%       1.175000
max      14.600000
Name: Year3_WS, dtype: float64

--- 範例名單 (前 5 名) ---
                Player  year_start  Year3_WS
220    DeMarre Carroll      2010.0       0.5
237        Jordan Hill      2010.0       1.4
244    Hasheem Thabeet      2010.0       0.3
245         Earl Clark      2010.0       0.3
257  Terrence Williams      2010.0       0.3


In [16]:
import pandas as pd

# 1. 讀取資料
# 請將 'NBA_Normalized_Final_3.csv' 替換成你實際的檔案路徑
file_name = 'NBA_Normalized_Final_3.csv'
df = pd.read_csv(file_name)

# 2. 定義需要從「總數」轉換為「單場平均」的欄位清單
cols_to_divide = [
    '2P', '2PA', '3P', '3PA', 'AST', 'BLK', 'DRB', 'FG', 'FGA', 
    'FT', 'FTA', 'MP', 'ORB', 'PF', 'PTS', 'STL', 'TOV', 'TRB'
]

# 3. 確保出賽場數 'G' 與目標欄位均為數值格式，避免運算錯誤
df['G'] = pd.to_numeric(df['G'], errors='coerce')

# 4. 執行除法運算：目標欄位 / G
for col in cols_to_divide:
    if col in df.columns:
        # 使用 pd.to_numeric 確保數據類型正確
        df[col] = pd.to_numeric(df[col], errors='coerce') / df['G']

# 5. 儲存更新後的檔案
output_name = 'NBA_Normalized_PerGame.csv'
df.to_csv(output_name, index=False)

print(f"處理完成！單場平均數據已儲存至: {output_name}")

處理完成！單場平均數據已儲存至: NBA_Normalized_PerGame.csv
